# 🫀 rPPG Transformer Pipeline — v5 FIXED (UBFC-rPPG)
### Complete Bug-Fix Release — All 12 Critical/High Issues Resolved

## 🐛 Bug Fix Summary

| # | Severity | Bug | Fix |
|---|----------|-----|-----|
| 1 | **CRITICAL** | GT resampled to `n_valid-1` assuming uniform face-detection — **timestamps misaligned** when frames skipped | Track video-frame indices; sample GT at exact detected-frame positions |
| 2 | **CRITICAL** | Phase-shift augmentation rolls signal ±3 frames without shifting frames — **100ms desync** | Removed phase-shift; only sync-safe augmentations retained |
| 3 | **CRITICAL** | `negative_pearson` computes mean over **all T including padded zeros** → biased gradient | Rewritten: mean computed over valid frames only |
| 4 | **CRITICAL** | `WARMUP_EPOCHS=5` with `EPOCHS=5` → LR only reaches full value at epoch 5, **model is in warmup the entire run** | `WARMUP_EPOCHS = max(1, EPOCHS//10)` — reaches cosine phase early |
| 5 | **CRITICAL** | Model outputs flat/near-zero signal → FFT always picks first BPM-band bin (42.19 BPM) → **constant MAE=43.74** | Fixed by bugs 1–4; added prediction-collapse diagnostic |
| 6 | **HIGH** | `enhanced_spectral_loss` / `snr_aware_loss` applied to zero-padded tail — **spurious DC + high-freq artefacts in FFT** | Both losses now mask out padding before FFT |
| 7 | **HIGH** | `detrend_signal()` builds N×N matrix and calls `np.linalg.solve` — **O(N³), too slow on CPU** | Replaced with `scipy.signal.detrend(type='linear')` |
| 8 | **HIGH** | `contrastive_temporal_loss` requires `B≥2` — returns **0.0 always when batch_size=1** (CPU) | Guard + `W_CONTRAST=0.0` when `BATCH_SIZE<2` |
| 9 | **HIGH** | `EMA_DECAY=0.999` too high for short CPU training → EMA barely updates | Adaptive decay: 0.99 on CPU, 0.999 on GPU |
| 10 | **MEDIUM** | `SEQ_LEN=256`, `IMG_SIZE=64`, `D_MODEL=128`, `NUM_LAYERS=4` → extremely slow on CPU | CPU-adaptive config: `SEQ_LEN=128`, `IMG_SIZE=32`, `D_MODEL=64`, `NUM_LAYERS=2` |
| 11 | **MEDIUM** | `snr_aware_loss` anti-trains when signal is random — pushes model toward flat output (cascade with Bug 5) | Loss disabled until epoch > warmup; weight capped at 0.05 |
| 12 | **MEDIUM** | `EPOCHS=5` on CPU — model has no chance to learn a complex dual-stream Transformer | Increased to 50 for CPU; added curriculum (Pearson-only first 5 epochs) |


In [7]:
# !pip install torch torchvision torchaudio
# !pip install mediapipe opencv-python-headless scipy numpy matplotlib tqdm
print("Dependencies ready.")


Dependencies ready.


In [8]:
import torch, platform, multiprocessing, os, math, random, time, hashlib, pickle, warnings
import numpy  as np
import torch.nn            as nn
import torch.nn.functional as F
import matplotlib.pyplot   as plt
import mediapipe           as mp
import csv

from pathlib               import Path
from tqdm                  import tqdm
from scipy.signal          import butter, filtfilt, detrend as scipy_detrend
from scipy.signal          import resample as scipy_resample
from scipy.stats           import pearsonr
from torch.utils.data      import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
from torch.cuda.amp        import GradScaler, autocast
from dataclasses           import dataclass

warnings.filterwarnings("ignore", category=UserWarning)
print(f"PyTorch {torch.__version__}")

print("=" * 60)
if torch.cuda.is_available():
    _DEVICE = "cuda"
    n_gpus  = torch.cuda.device_count()
    print(f"  GPU Available — {n_gpus} device(s)")
    for i in range(n_gpus):
        p   = torch.cuda.get_device_properties(i)
        mem = p.total_memory / (1024**3)
        print(f"  GPU {i}: {p.name}  {mem:.1f}GB  compute {p.major}.{p.minor}")
    torch.backends.cudnn.benchmark     = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32    = True
    print(f"  PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}")
    print("  ► Training on GPU — TF32 enabled")
else:
    _DEVICE = "cpu"
    cores   = multiprocessing.cpu_count()
    print(f"  No GPU — falling back to CPU")
    print(f"  Cores: {cores}  Platform: {platform.processor() or platform.machine()}")
    torch.set_num_threads(cores)
    print(f"  PyTorch threads: {torch.get_num_threads()}")
    print("  ► Training on CPU (use Google Colab for GPU)")
print("=" * 60)


PyTorch 2.3.1+cpu
  No GPU — falling back to CPU
  Cores: 40  Platform: Intel64 Family 6 Model 143 Stepping 8, GenuineIntel
  PyTorch threads: 40
  ► Training on CPU (use Google Colab for GPU)


In [9]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)
print("Seed set to 42.")


Seed set to 42.


## 🛠️ Configuration (v5 — CPU-Adaptive)

### Key changes from v4
| Parameter | v4 (CPU) | v5 (CPU) | Why |
|-----------|----------|----------|-----|
| `SEQ_LEN` | 256 | **128** | 4× faster Transformer; still covers 4.3s |
| `IMG_SIZE` | 64 | **32** | 4× faster CNN per frame |
| `D_MODEL` | 128 | **64** | 4× smaller model, faster convergence |
| `NUM_LAYERS` | 4 | **2** | 2× faster Transformer |
| `NHEAD` | 8 | **4** | matches D_MODEL=64 |
| `EPOCHS` | 5 | **50** | model needs epochs to learn |
| `WARMUP_EPOCHS` | 5 | **3** | FIX #4: leaves room for cosine annealing |
| `EMA_DECAY` | 0.999 | **0.99** | FIX #9: faster EMA tracking on CPU |
| `W_CONTRAST` | 0.1 | **0.0** | FIX #8: useless with B=1 |
| `W_SNR` | 0.1 | **0.05** | FIX #11: reduced to prevent signal collapse |


In [10]:
@dataclass
class CFG:
    # ── Device ──────────────────────────────────────────────────────────
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    # ── Model — CPU-adaptive sizes ───────────────────────────────────────
    # FIX #10: smaller model for CPU (4× faster CNN + 4× faster Transformer)
    IMG_SIZE:   int   = 64  if torch.cuda.is_available() else 32   # 64→32 on CPU
    SEQ_LEN:    int   = 256 if torch.cuda.is_available() else 128  # 256→128 on CPU
    D_MODEL:    int   = 128 if torch.cuda.is_available() else 64   # 128→64 on CPU
    NHEAD:      int   = 8   if torch.cuda.is_available() else 4    # 8→4 on CPU
    NUM_LAYERS: int   = 4   if torch.cuda.is_available() else 2    # 4→2 on CPU
    DIM_FF:     int   = 512 if torch.cuda.is_available() else 256  # 512→256 on CPU
    DROPOUT:    float = 0.1

    # ── Training ────────────────────────────────────────────────────────
    BATCH_SIZE:    int   = 4   if torch.cuda.is_available() else 1
    # FIX #12: more epochs so model can learn; was 5 on CPU
    EPOCHS:        int   = 100 if torch.cuda.is_available() else 50
    LR:            float = 3e-4
    WEIGHT_DECAY:  float = 1e-2
    ACCUM_STEPS:   int   = 4    # effective batch = 4 on CPU
    GRAD_CLIP:     float = 1.0
    # FIX #4: warmup = 10% of epochs so cosine phase is actually reached
    WARMUP_EPOCHS: int   = 10  if torch.cuda.is_available() else 3
    # FIX #9: lower EMA decay for CPU (fewer steps total)
    EMA_DECAY:     float = 0.999 if torch.cuda.is_available() else 0.99
    EARLY_STOP:    int   = 20   # patience in epochs
    USE_AMP:       bool  = torch.cuda.is_available()
    GRAD_CKPT:     bool  = torch.cuda.is_available()

    # ── Loss weights ────────────────────────────────────────────────────
    W_PEARSON:  float = 0.5   # primary signal alignment loss
    W_SMOOTH:   float = 0.3   # amplitude regression
    W_SPECTRAL: float = 0.15  # frequency-domain alignment
    # FIX #11: SNR loss reduced — was pushing model to flat output
    W_SNR:      float = 0.05
    # FIX #8: contrastive disabled when batch_size=1 (CPU)
    W_CONTRAST: float = 0.0   if not torch.cuda.is_available() else 0.05

    # ── Data ────────────────────────────────────────────────────────────
    FPS:         float = 30.0
    MIN_FRAMES:  int   = 64
    NUM_WORKERS: int   = 0     # 0 = safe on Windows
    USE_CACHE:   bool  = True
    STRIDE:      int   = 32    # sliding-window stride (was 64 — more windows)

    # ── UBFC-rPPG ───────────────────────────────────────────────────────
    # ↓  CHANGE THIS before running
    UBFC_ROOT:   str   = r"C:\\Users\\UIET\\Desktop\\physformer\\dataset"
    GT_FILENAME: str   = "ground_truth.txt"
    TRAIN_RATIO: float = 0.8

    # ── Checkpoints ─────────────────────────────────────────────────────
    CKPT_DIR:    str   = "./checkpoints_v5"
    SAVE_BEST:   str   = "best_rppg_v5.pth"
    SAVE_FINAL:  str   = "final_rppg_v5.pth"
    LOG_FILE:    str   = "train_log_v5.csv"

cfg = CFG()
os.makedirs(cfg.CKPT_DIR, exist_ok=True)

print(f"{'='*60}")
print(f"  CFG loaded  |  Device : {cfg.DEVICE.upper()}")
print(f"  SEQ_LEN     : {cfg.SEQ_LEN}  |  IMG_SIZE   : {cfg.IMG_SIZE}")
print(f"  D_MODEL     : {cfg.D_MODEL}  |  NHEAD      : {cfg.NHEAD}")
print(f"  NUM_LAYERS  : {cfg.NUM_LAYERS}  |  DIM_FF     : {cfg.DIM_FF}")
print(f"  EPOCHS      : {cfg.EPOCHS}  |  WARMUP     : {cfg.WARMUP_EPOCHS}")
print(f"  BATCH_SIZE  : {cfg.BATCH_SIZE}  |  ACCUM      : {cfg.ACCUM_STEPS}")
print(f"  EMA_DECAY   : {cfg.EMA_DECAY}  |  USE_AMP    : {cfg.USE_AMP}")
print(f"  W_PEARSON   : {cfg.W_PEARSON} |  W_SMOOTH   : {cfg.W_SMOOTH}")
print(f"  W_SPECTRAL  : {cfg.W_SPECTRAL}| W_SNR      : {cfg.W_SNR}")
print(f"  W_CONTRAST  : {cfg.W_CONTRAST}")
print(f"  UBFC_ROOT   : {cfg.UBFC_ROOT}")
print(f"{'='*60}")


  CFG loaded  |  Device : CPU
  SEQ_LEN     : 128  |  IMG_SIZE   : 32
  D_MODEL     : 64  |  NHEAD      : 4
  NUM_LAYERS  : 2  |  DIM_FF     : 256
  EPOCHS      : 50  |  WARMUP     : 3
  BATCH_SIZE  : 1  |  ACCUM      : 4
  EMA_DECAY   : 0.99  |  USE_AMP    : False
  W_PEARSON   : 0.5 |  W_SMOOTH   : 0.3
  W_SPECTRAL  : 0.15| W_SNR      : 0.05
  W_CONTRAST  : 0.0
  UBFC_ROOT   : C:\\Users\\UIET\\Desktop\\physformer\\dataset


In [11]:
# MediaPipe landmark groups for three skin ROIs
FOREHEAD_IDXS = [10, 67, 103, 109, 338, 297, 332, 333, 334, 296]
LEFT_CHEEK    = [50, 187, 205, 207, 216, 206, 203]
RIGHT_CHEEK   = [280, 425, 411, 427, 436, 426, 423]

_face_mesh_registry: dict = {}

def get_face_mesh() -> mp.solutions.face_mesh.FaceMesh:
    pid = os.getpid()
    if pid not in _face_mesh_registry:
        _face_mesh_registry[pid] = mp.solutions.face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5,   # lowered from 0.7 — fewer skipped frames
            min_tracking_confidence=0.5,
        )
    return _face_mesh_registry[pid]

def worker_init_fn(worker_id: int) -> None:
    set_seed(42 + worker_id)
    get_face_mesh()

print("FaceMesh registry ready.  Detection threshold lowered to 0.5 (fewer skipped frames).")


FaceMesh registry ready.  Detection threshold lowered to 0.5 (fewer skipped frames).


## 📈 Signal Processing
**FIX #7:** Replaced O(N³) `detrend_signal()` with `scipy.signal.detrend` (O(N)).  
BPM band kept at 45–150 BPM (standard for UBFC-rPPG; avoids unlikely extremes).


In [12]:
# # ── BPM / frequency constants ────────────────────────────────────────────
# # Standard physiological range for UBFC-rPPG subjects
# BPM_LOW,  BPM_HIGH  = 45.0,  150.0
# FREQ_LOW, FREQ_HIGH = BPM_LOW/60, BPM_HIGH/60

# def detrend_signal(signal: np.ndarray) -> np.ndarray:
#     \"\"\"
#     FIX #7: Use scipy.signal.detrend (O(N)) instead of the O(N³)
#     smoothness-priors matrix solve that was in v4.
#     Removes linear + slow baseline drift.
#     \"\"\"
#     if len(signal) < 4:
#         return signal
#     return scipy_detrend(signal, type='linear').astype(np.float32)


# def bandpass_filter(
#     signal: np.ndarray,
#     fs:     float = 30.0,
#     lo:     float = FREQ_LOW,
#     hi:     float = FREQ_HIGH,
#     order:  int   = 4,
# ) -> np.ndarray:
#     \"\"\"
#     Butterworth bandpass with automatic order reduction for short signals.
#     Returns z-score normalised filtered signal.
#     \"\"\"
#     nyq = fs * 0.5
#     lo, hi = max(lo, 0.01), min(hi, nyq * 0.99)
#     if lo >= hi:
#         s = signal - signal.mean()
#         return (s / (s.std() + 1e-8)).astype(np.float32)

#     effective_order = order
#     while len(signal) < 3 * (2 * effective_order + 1) and effective_order > 1:
#         effective_order -= 1

#     if len(signal) < 15:
#         s = signal - signal.mean()
#         return (s / (s.std() + 1e-8)).astype(np.float32)

#     b, a  = butter(effective_order, [lo/nyq, hi/nyq], btype='band')
#     filt  = filtfilt(b, a, signal)
#     return ((filt - filt.mean()) / (filt.std() + 1e-8)).astype(np.float32)


# def compute_bpm(signal: np.ndarray, fs: float = 30.0) -> float:
#     \"\"\"Dominant BPM via FFT within physiological band [45,150] BPM.\"\"\"
#     if len(signal) < 8:
#         return 75.0
#     signal  = bandpass_filter(signal, fs)
#     freqs   = np.fft.rfftfreq(len(signal), d=1.0/fs)
#     fft_mag = np.abs(np.fft.rfft(signal * np.hanning(len(signal))))
#     mask    = (freqs >= FREQ_LOW) & (freqs <= FREQ_HIGH)
#     if mask.sum() == 0:
#         return 75.0
#     return float(freqs[mask][np.argmax(fft_mag[mask])] * 60.0)


# def compute_snr(signal: np.ndarray, fs: float = 30.0) -> float:
#     \"\"\"SNR = 10*log10(in-band power / out-of-band power).\"\"\"
#     if len(signal) < 8:
#         return -99.0
#     freqs   = np.fft.rfftfreq(len(signal), d=1.0/fs)
#     fft_pow = np.abs(np.fft.rfft(signal * np.hanning(len(signal)))) ** 2
#     band    = (freqs >= FREQ_LOW) & (freqs <= FREQ_HIGH)
#     p_sig   = fft_pow[band].sum()  + 1e-12
#     p_noise = fft_pow[~band].sum() + 1e-12
#     return float(10 * np.log10(p_sig / p_noise))


# print(f"Signal processing ready.  BPM band: [{BPM_LOW},{BPM_HIGH}]  Freq: [{FREQ_LOW:.3f},{FREQ_HIGH:.3f}] Hz")


# ── BPM / Frequency Constants ─────────────────────────────────────────────
# Standard physiological heart-rate range for UBFC-rPPG subjects
BPM_LOW: float = 45.0
BPM_HIGH: float = 150.0

# Convert BPM range to frequency range in Hz
FREQ_LOW: float = BPM_LOW / 60.0
FREQ_HIGH: float = BPM_HIGH / 60.0


def detrend_signal(signal: np.ndarray) -> np.ndarray:
    """
    Remove linear baseline drift from the signal.

    Uses scipy.signal.detrend(), which is computationally efficient
    compared to smoothness-prior matrix approaches.

    Args:
        signal: Input 1D signal array.

    Returns:
        Detrended signal as float32.
    """
    if len(signal) < 4:
        return signal.astype(np.float32)

    return scipy_detrend(signal, type="linear").astype(np.float32)


def bandpass_filter(
    signal: np.ndarray,
    fs: float = 30.0,
    lo: float = FREQ_LOW,
    hi: float = FREQ_HIGH,
    order: int = 4,
) -> np.ndarray:
    """
    Apply a Butterworth bandpass filter and normalize the result.

    Features:
    - Automatically reduces filter order for short signals.
    - Falls back to z-score normalization if filtering is unstable.
    - Output is standardized (zero mean, unit variance).

    Args:
        signal: Input 1D signal.
        fs: Sampling frequency in Hz.
        lo: Lower cutoff frequency in Hz.
        hi: Upper cutoff frequency in Hz.
        order: Initial Butterworth filter order.

    Returns:
        Filtered and normalized signal as float32.
    """
    nyq = fs * 0.5

    # Clamp cutoff frequencies to valid range
    lo = max(lo, 0.01)
    hi = min(hi, nyq * 0.99)

    # Fallback if frequency range is invalid
    if lo >= hi:
        s = signal - signal.mean()
        return (s / (s.std() + 1e-8)).astype(np.float32)

    # Reduce filter order for short signals
    effective_order = order
    while len(signal) < 3 * (2 * effective_order + 1) and effective_order > 1:
        effective_order -= 1

    # Very short signals are unsafe for filtfilt()
    if len(signal) < 15:
        s = signal - signal.mean()
        return (s / (s.std() + 1e-8)).astype(np.float32)

    # Design and apply Butterworth bandpass filter
    b, a = butter(
        effective_order,
        [lo / nyq, hi / nyq],
        btype="band",
    )

    filt = filtfilt(b, a, signal)

    # Z-score normalization
    filt = (filt - filt.mean()) / (filt.std() + 1e-8)

    return filt.astype(np.float32)


def compute_bpm(signal: np.ndarray, fs: float = 30.0) -> float:
    """
    Estimate heart rate (BPM) using FFT peak detection.

    Steps:
    1. Bandpass filter the signal.
    2. Compute FFT magnitude spectrum.
    3. Find dominant frequency within physiological range.
    4. Convert frequency (Hz) to BPM.

    Args:
        signal: Input 1D signal.
        fs: Sampling frequency in Hz.

    Returns:
        Estimated BPM value.
    """
    if len(signal) < 8:
        return 75.0

    signal = bandpass_filter(signal, fs)

    freqs = np.fft.rfftfreq(len(signal), d=1.0 / fs)

    fft_mag = np.abs(
        np.fft.rfft(signal * np.hanning(len(signal)))
    )

    # Physiological frequency mask
    mask = (freqs >= FREQ_LOW) & (freqs <= FREQ_HIGH)

    if mask.sum() == 0:
        return 75.0

    dominant_freq = freqs[mask][np.argmax(fft_mag[mask])]

    return float(dominant_freq * 60.0)


def compute_snr(signal: np.ndarray, fs: float = 30.0) -> float:
    """
    Compute signal-to-noise ratio (SNR) in dB.

    SNR = 10 * log10(signal_band_power / noise_power)

    Signal power is measured inside the physiological
    heart-rate band, while noise power is measured outside it.

    Args:
        signal: Input 1D signal.
        fs: Sampling frequency in Hz.

    Returns:
        SNR value in decibels (dB).
    """
    if len(signal) < 8:
        return -99.0

    freqs = np.fft.rfftfreq(len(signal), d=1.0 / fs)

    fft_pow = np.abs(
        np.fft.rfft(signal * np.hanning(len(signal)))
    ) ** 2

    # In-band physiological frequencies
    band = (freqs >= FREQ_LOW) & (freqs <= FREQ_HIGH)

    p_sig = fft_pow[band].sum() + 1e-12
    p_noise = fft_pow[~band].sum() + 1e-12

    return float(10.0 * np.log10(p_sig / p_noise))


print(
    f"Signal processing ready | "
    f"BPM range: [{BPM_LOW:.1f}, {BPM_HIGH:.1f}] | "
    f"Frequency range: [{FREQ_LOW:.3f}, {FREQ_HIGH:.3f}] Hz"
)

Signal processing ready | BPM range: [45.0, 150.0] | Frequency range: [0.750, 2.500] Hz


In [15]:
class LandmarkStabiliser:
    # "\"\"EMA smoothing over landmark positions to reduce jitter.\"\"\"
    def __init__(self, alpha: float = 0.6):
        self.alpha   = alpha
        self.prev    = {}

    def smooth(self, idx: int, pts: np.ndarray) -> np.ndarray:
        if idx not in self.prev:
            self.prev[idx] = pts.astype(np.float32)
            return pts
        s = self.alpha * pts + (1 - self.alpha) * self.prev[idx]
        self.prev[idx] = s
        return s.astype(np.int32)


def skin_mask_ycrcb(roi_bgr: np.ndarray) -> np.ndarray:
    ycrcb = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2YCrCb)
    mask  = cv2.inRange(ycrcb,
                        np.array([0, 130, 75],  dtype=np.uint8),
                        np.array([255,180,135], dtype=np.uint8))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return   cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)


def clahe_normalise(roi_rgb: np.ndarray) -> np.ndarray:
    lab  = cv2.cvtColor(roi_rgb, cv2.COLOR_RGB2LAB)
    c    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4,4))
    lab[:,:,0] = c.apply(lab[:,:,0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def extract_roi(frame_rgb, landmarks, indices,
                stabiliser=None, roi_idx=0,
                apply_skin_mask=True, apply_clahe=True,
                img_size=32):
    h, w, _ = frame_rgb.shape
    pts = np.array([(int(landmarks.landmark[i].x*w),
                     int(landmarks.landmark[i].y*h)) for i in indices], dtype=np.int32)
    if stabiliser is not None:
        pts = stabiliser.smooth(roi_idx, pts)

    hull     = cv2.convexHull(pts)
    geo_mask = np.zeros((h,w), dtype=np.uint8)
    cv2.fillConvexPoly(geo_mask, hull, 255)
    x, y, bw, bh = cv2.boundingRect(hull)
    pad = max(2, min(bw,bh)//8)
    x  = max(0, x-pad);  y  = max(0, y-pad)
    bw = min(w-x, bw+2*pad); bh = min(h-y, bh+2*pad)

    if bw < 5 or bh < 5:
        return None, None

    roi_rgb   = frame_rgb[y:y+bh, x:x+bw].copy()
    mask_crop = geo_mask[y:y+bh, x:x+bw]

    if apply_clahe:
        roi_rgb = clahe_normalise(roi_rgb)

    if apply_skin_mask:
        roi_bgr  = cv2.cvtColor(roi_rgb, cv2.COLOR_RGB2BGR)
        skin_m   = skin_mask_ycrcb(roi_bgr)
        combined = cv2.bitwise_and(mask_crop, skin_m)
    else:
        combined = mask_crop

    skin_pixels = roi_rgb[combined > 0]
    rgb_mean = (skin_pixels.mean(axis=0).astype(np.float32)
                if len(skin_pixels) >= 10
                else roi_rgb.reshape(-1,3).mean(axis=0).astype(np.float32))

    roi_resized = cv2.resize(roi_rgb, (img_size, img_size)).astype(np.float32) / 255.0
    return roi_resized, rgb_mean


print("ROI extraction ready.  CLAHE + YCrCb skin mask + convex-hull bounding.")


ROI extraction ready.  CLAHE + YCrCb skin mask + convex-hull bounding.


## 🎲 Augmentation Pipeline (v5 — Phase Shift REMOVED)
**FIX #2:** Phase-shift augmentation removed — it desynced signal from frames by ±3 frames
(100ms at 30fps, comparable to a heartbeat width), corrupting the learning signal.

All remaining augmentations are frame-sync-safe.


In [16]:
import numpy as np
from scipy.signal import butter, filtfilt, detrend as scipy_detrend


# ──────────────────────────────────────────────────────────────────────────
# BPM / Frequency Constants
# Standard physiological heart-rate range for UBFC-rPPG subjects
# ──────────────────────────────────────────────────────────────────────────

BPM_LOW: float = 45.0
BPM_HIGH: float = 150.0

# Convert BPM range to frequency range in Hz
FREQ_LOW: float = BPM_LOW / 60.0
FREQ_HIGH: float = BPM_HIGH / 60.0


def detrend_signal(signal: np.ndarray) -> np.ndarray:
    """
    Remove slow linear baseline drift from a signal.

    Uses scipy.signal.detrend(), which is efficient and stable.

    Parameters
    ----------
    signal : np.ndarray
        Input 1D signal.

    Returns
    -------
    np.ndarray
        Detrended signal in float32 format.
    """
    signal = np.asarray(signal, dtype=np.float32)

    if signal.size < 4:
        return signal

    return scipy_detrend(signal, type="linear").astype(np.float32)


def zscore_normalize(signal: np.ndarray) -> np.ndarray:
    """
    Apply z-score normalization.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.

    Returns
    -------
    np.ndarray
        Normalized signal.
    """
    signal = signal - np.mean(signal)
    std = np.std(signal)

    if std < 1e-8:
        return np.zeros_like(signal, dtype=np.float32)

    return (signal / std).astype(np.float32)


def bandpass_filter(
    signal: np.ndarray,
    fs: float = 30.0,
    lo: float = FREQ_LOW,
    hi: float = FREQ_HIGH,
    order: int = 4,
) -> np.ndarray:
    """
    Apply Butterworth bandpass filtering.

    Features
    --------
    - Automatically reduces filter order for short signals.
    - Prevents invalid cutoff frequencies.
    - Uses zero-phase filtering via filtfilt().
    - Returns z-score normalized output.

    Parameters
    ----------
    signal : np.ndarray
        Input 1D signal.
    fs : float
        Sampling frequency in Hz.
    lo : float
        Lower cutoff frequency in Hz.
    hi : float
        Upper cutoff frequency in Hz.
    order : int
        Initial Butterworth filter order.

    Returns
    -------
    np.ndarray
        Filtered and normalized signal.
    """
    signal = np.asarray(signal, dtype=np.float32)

    if signal.size < 15:
        return zscore_normalize(signal)

    nyquist = 0.5 * fs

    # Clamp cutoff frequencies to valid range
    lo = max(lo, 0.01)
    hi = min(hi, nyquist * 0.99)

    # Handle invalid frequency range
    if lo >= hi:
        return zscore_normalize(signal)

    # Reduce filter order for short signals
    effective_order = order

    while (
        signal.size < 3 * (2 * effective_order + 1)
        and effective_order > 1
    ):
        effective_order -= 1

    # Design Butterworth bandpass filter
    b, a = butter(
        effective_order,
        [lo / nyquist, hi / nyquist],
        btype="band",
    )

    try:
        filtered = filtfilt(b, a, signal)
    except ValueError:
        # Fallback if filtering becomes unstable
        return zscore_normalize(signal)

    return zscore_normalize(filtered)


def compute_bpm(signal: np.ndarray, fs: float = 30.0) -> float:
    """
    Estimate heart rate (BPM) using FFT peak detection.

    Workflow
    --------
    1. Bandpass filter the signal.
    2. Apply Hann window.
    3. Compute FFT magnitude spectrum.
    4. Find dominant frequency in physiological range.
    5. Convert frequency to BPM.

    Parameters
    ----------
    signal : np.ndarray
        Input 1D signal.
    fs : float
        Sampling frequency in Hz.

    Returns
    -------
    float
        Estimated heart rate in BPM.
    """
    signal = np.asarray(signal, dtype=np.float32)

    if signal.size < 8:
        return 75.0

    signal = bandpass_filter(signal, fs)

    windowed = signal * np.hanning(signal.size)

    freqs = np.fft.rfftfreq(signal.size, d=1.0 / fs)
    fft_mag = np.abs(np.fft.rfft(windowed))

    # Keep only physiological heart-rate frequencies
    mask = (freqs >= FREQ_LOW) & (freqs <= FREQ_HIGH)

    if not np.any(mask):
        return 75.0

    dominant_freq = freqs[mask][np.argmax(fft_mag[mask])]

    return float(dominant_freq * 60.0)


def compute_snr(signal: np.ndarray, fs: float = 30.0) -> float:
    """
    Compute signal-to-noise ratio (SNR) in decibels.

    Signal power is measured within the physiological
    heart-rate frequency band.

    Parameters
    ----------
    signal : np.ndarray
        Input 1D signal.
    fs : float
        Sampling frequency in Hz.

    Returns
    -------
    float
        SNR value in dB.
    """
    signal = np.asarray(signal, dtype=np.float32)

    if signal.size < 8:
        return -99.0

    windowed = signal * np.hanning(signal.size)

    freqs = np.fft.rfftfreq(signal.size, d=1.0 / fs)
    fft_power = np.abs(np.fft.rfft(windowed)) ** 2

    # Physiological frequency band
    band_mask = (freqs >= FREQ_LOW) & (freqs <= FREQ_HIGH)

    signal_power = np.sum(fft_power[band_mask]) + 1e-12
    noise_power = np.sum(fft_power[~band_mask]) + 1e-12

    snr_db = 10.0 * np.log10(signal_power / noise_power)

    return float(snr_db)


print(
    f"Signal processing ready | "
    f"BPM range: [{BPM_LOW:.1f}, {BPM_HIGH:.1f}] | "
    f"Frequency range: [{FREQ_LOW:.3f}, {FREQ_HIGH:.3f}] Hz"
)

Signal processing ready | BPM range: [45.0, 150.0] | Frequency range: [0.750, 2.500] Hz


## 📊 UBFC Ground-Truth Loader & Video Processor

### FIX #1 — GT Frame-Index Alignment (Most Critical)

**Problem in v4:**  
`load_ubfc_gt(gt_path, target_length=n_valid-1)` calls `scipy_resample(bvp, n_valid-1)`.  
This assumes detected frames are **uniformly spaced** in time — but they are not!  
Whenever MediaPipe fails to detect a face, that video frame is **skipped**.  
The GT file has one entry per **video frame**, not per detected frame.  
Resampling to `n_valid-1` creates a misaligned signal.

**Fix:**  
`process_video` now returns `frame_indices` — the list of video-frame numbers  
where face detection succeeded. `load_ubfc_gt_indexed` samples the GT array  
at exactly those indices, giving true temporal alignment.

```
Video frames   :  0  1  2  3 [4 skipped]  5  6  7 [8 skipped]  9 ...
GT array       : g0 g1 g2 g3      —       g5 g6 g7      —       g9 ...
Detected frames:  0  1  2  3               5  6  7                9 ...

v4 (WRONG): scipy_resample(gt, n_valid) → uniform spacing, ignores gaps
v5 (FIXED): gt[frame_indices]           → exact timestamp matching ✓
```


In [17]:
import os
import cv2
import pickle
import hashlib
import numpy as np

from pathlib import Path
from scipy.signal import detrend as scipy_detrend
from scipy.signal import butter, filtfilt
from scipy.signal import resample as scipy_resample


def load_ubfc_gt_indexed(
    gt_path: str,
    frame_indices: list,
    total_frames: int,
    fs: float = 30.0,
) -> np.ndarray:
    """
    Load and align UBFC ground-truth BVP signal to detected video frames.

    The GT signal is:
    1. Loaded from ground_truth.txt
    2. Resampled to match total video frame count
    3. Sampled using exact detected-frame indices
    4. Temporally aligned with motion-difference frames
    5. Detrended and bandpass filtered

    Parameters
    ----------
    gt_path : str
        Path to UBFC ground_truth.txt file.

    frame_indices : list
        Video-frame indices where valid faces were detected.

    total_frames : int
        Total number of frames in the original video.

    fs : float, default=30.0
        Sampling frequency in Hz.

    Returns
    -------
    np.ndarray
        Processed BVP signal aligned to diff-frame sequence.
    """
    if not os.path.isfile(gt_path):
        raise FileNotFoundError(f"GT file not found: {gt_path}")

    # Load raw ground-truth signal
    raw = np.loadtxt(gt_path, dtype=np.float64)

    if raw.ndim == 2:
        bvp = raw[:, 0].astype(np.float32)
    else:
        bvp = raw.astype(np.float32)

    # Resample GT signal to match video frame count
    if len(bvp) != total_frames:
        bvp = scipy_resample(bvp, total_frames).astype(np.float32)

    # Align GT using detected frame positions
    indices = np.asarray(frame_indices, dtype=np.int64)
    indices = np.clip(indices, 0, len(bvp) - 1)

    bvp_aligned = bvp[indices]

    # Match temporal-difference stream length (N-1)
    if len(bvp_aligned) > 1:
        bvp_aligned = bvp_aligned[:-1]

    # Detrend signal
    if len(bvp_aligned) > 8:
        bvp_aligned = detrend_signal(bvp_aligned)

    # Bandpass filtering + normalization
    signal = bandpass_filter(bvp_aligned, fs=fs)

    return signal.astype(np.float32)


def _cache_path(video_path: str, img_size: int) -> Path:
    """
    Generate deterministic cache path for processed video data.
    """
    cache_key = f"{video_path}_{img_size}_v5"

    cache_hash = hashlib.sha256(
        cache_key.encode()
    ).hexdigest()[:12]

    return Path(cfg.CKPT_DIR) / f"cache_{cache_hash}.pkl"


def process_video(
    video_path: str,
    use_cache: bool = True,
) -> tuple:
    """
    Extract dual-stream face data from a video.

    Returns:
    --------
    diff_frames : np.ndarray
        Normalized temporal-difference frames.

    raw_frames : np.ndarray
        Raw appearance frames.

    frame_indices : list[int]
        Original video-frame indices where face detection succeeded.

    total_frames : int
        Total frames read from the video.

    n_valid : int
        Number of valid face-detected frames.
    """
    if not os.path.isfile(video_path):
        raise FileNotFoundError(
            f"Video file not found: {video_path}"
        )

    # ---------------------------------------------------------------------
    # Cache loading
    # ---------------------------------------------------------------------
    cache_file = _cache_path(video_path, cfg.IMG_SIZE)

    if use_cache and cache_file.exists():
        try:
            with open(cache_file, "rb") as f:
                return pickle.load(f)

        except Exception:
            cache_file.unlink(missing_ok=True)

    # ---------------------------------------------------------------------
    # Video initialization
    # ---------------------------------------------------------------------
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError(
            f"Unable to open video: {video_path}"
        )

    face_mesh = get_face_mesh()
    stabiliser = LandmarkStabiliser(alpha=0.65)

    raw_frames_list = []
    frame_indices = []

    frame_count = 0

    # ROI landmark groups
    roi_groups = [
        FOREHEAD_IDXS,
        LEFT_CHEEK,
        RIGHT_CHEEK,
    ]

    # ---------------------------------------------------------------------
    # Frame processing loop
    # ---------------------------------------------------------------------
    while True:
        ret, frame_bgr = cap.read()

        if not ret:
            break

        frame_rgb = cv2.cvtColor(
            frame_bgr,
            cv2.COLOR_BGR2RGB,
        )

        results = face_mesh.process(frame_rgb)

        if results.multi_face_landmarks:

            landmarks = results.multi_face_landmarks[0]

            roi_images = []

            # Extract all facial ROIs
            for roi_idx, landmark_indices in enumerate(roi_groups):

                roi_img, _ = extract_roi(
                    frame_rgb,
                    landmarks,
                    landmark_indices,
                    stabiliser=stabiliser,
                    roi_idx=roi_idx,
                    img_size=cfg.IMG_SIZE,
                )

                if roi_img is not None:
                    roi_images.append(roi_img)

            # Merge ROI crops into single appearance frame
            if roi_images:

                face_img = np.mean(
                    np.stack(roi_images, axis=0),
                    axis=0,
                ).astype(np.float32)

                raw_frames_list.append(face_img)

                # Save original video-frame index
                frame_indices.append(frame_count)

        frame_count += 1

    cap.release()

    total_frames = frame_count
    n_valid = len(raw_frames_list)

    # ---------------------------------------------------------------------
    # Validation
    # ---------------------------------------------------------------------
    if n_valid < cfg.MIN_FRAMES:
        raise ValueError(
            f"Only {n_valid} valid frames found in "
            f"{Path(video_path).name} "
            f"(minimum required = {cfg.MIN_FRAMES})"
        )

    # ---------------------------------------------------------------------
    # Convert to arrays
    # ---------------------------------------------------------------------
    raw_arr = np.asarray(
        raw_frames_list,
        dtype=np.float32,
    )

    # Temporal-difference motion stream
    diff_arr = raw_arr[1:] - raw_arr[:-1]

    # Normalize motion stream
    mean = diff_arr.mean(
        axis=(0, 1, 2),
        keepdims=True,
    )

    std = diff_arr.std(
        axis=(0, 1, 2),
        keepdims=True,
    )

    diff_arr = (diff_arr - mean) / (std + 1e-8)

    # Align raw appearance stream to diff-stream length
    raw_arr = raw_arr[:-1]

    data = (
        diff_arr.astype(np.float32),
        raw_arr.astype(np.float32),
        frame_indices,
        total_frames,
        n_valid,
    )

    # ---------------------------------------------------------------------
    # Cache save
    # ---------------------------------------------------------------------
    if use_cache:
        try:
            with open(cache_file, "wb") as f:
                pickle.dump(
                    data,
                    f,
                    protocol=pickle.HIGHEST_PROTOCOL,
                )

        except Exception:
            pass

    return data


print("Video processor + GT loader ready.")
print("Exact GT alignment using detected frame indices is enabled.")

Video processor + GT loader ready.
Exact GT alignment using detected frame indices is enabled.


## 🪟 Dataset (v5 — Fixed Window Generation)
Uses `load_ubfc_gt_indexed` with exact frame-index sampling (FIX #1).  
Sliding-window stride reduced from 64 → 32 to **generate more training samples**.


In [18]:
from pathlib import Path

import numpy as np
import torch

from torch.utils.data import Dataset


def pad_or_crop(
    arr: np.ndarray,
    length: int,
    axis: int = 0,
) -> np.ndarray:
    """
    Pad or crop an array along a specified axis.

    If the array length exceeds the target length,
    it is cropped. Otherwise, zero-padding is applied.

    Parameters
    ----------
    arr : np.ndarray
        Input array.

    length : int
        Target length.

    axis : int, default=0
        Axis along which padding/cropping is applied.

    Returns
    -------
    np.ndarray
        Output array with fixed length.
    """
    arr = np.asarray(arr)

    current_length = arr.shape[axis]

    # Crop if array is longer than target
    if current_length >= length:
        indices = range(length)
        return np.take(arr, indices, axis=axis)

    # Pad with zeros if array is shorter
    pad_width = [(0, 0)] * arr.ndim
    pad_width[axis] = (0, length - current_length)

    return np.pad(
        arr,
        pad_width,
        mode="constant",
        constant_values=0.0,
    )


class RPPGDataset(Dataset):
    """
    UBFC-rPPG supervised dataset with alignment fixes.

    Features
    --------
    - Exact GT alignment using detected frame indices.
    - Temporal-difference motion stream support.
    - Sliding-window sampling for better coverage.
    - Optional cached preprocessing.
    - Padding support for variable-length sequences.

    Notes
    -----
    - Phase-shift augmentation has been removed.
    - Temporal difference reduces frame count by 1.
    """

    def __init__(
        self,
        video_paths,
        gt_paths,
        training: bool = True,
        use_cache: bool = True,
    ):
        """
        Initialize dataset.

        Parameters
        ----------
        video_paths : list
            List of video file paths.

        gt_paths : list
            List of ground-truth signal paths.

        training : bool, default=True
            Enables data augmentation if True.

        use_cache : bool, default=True
            Enables cached preprocessing.
        """
        assert len(video_paths) == len(gt_paths), (
            "video_paths and gt_paths must have same length"
        )

        self.training = training
        self.use_cache = use_cache

        # Window entries:
        # (
        #   video_path,
        #   gt_path,
        #   frame_indices,
        #   total_frames,
        #   start_index
        # )
        self.windows = []

        dataset_type = "train" if training else "val"

        print(f"Building {dataset_type} dataset...")

        skipped = 0

        for video_path, gt_path in zip(video_paths, gt_paths):

            try:
                (
                    _,
                    _,
                    frame_indices,
                    total_frames,
                    n_valid,
                ) = process_video(
                    video_path,
                    use_cache=use_cache,
                )

                # Temporal difference reduces length by 1
                n_frames = n_valid - 1

                # Sliding-window generation
                max_start = max(
                    1,
                    n_frames - cfg.SEQ_LEN + 1,
                )

                starts = list(
                    range(
                        0,
                        max_start,
                        cfg.STRIDE,
                    )
                )

                if not starts:
                    starts = [0]

                for start_idx in starts:

                    self.windows.append(
                        (
                            video_path,
                            gt_path,
                            frame_indices,
                            total_frames,
                            start_idx,
                        )
                    )

            except (
                ValueError,
                FileNotFoundError,
            ) as e:

                print(
                    f"  Skip {Path(video_path).parent.name}: {e}"
                )

                skipped += 1

        valid_videos = len(video_paths) - skipped

        print(
            f"  -> {valid_videos} videos | "
            f"{len(self.windows)} windows "
            f"(SEQ_LEN={cfg.SEQ_LEN}, "
            f"STRIDE={cfg.STRIDE})"
        )

    def __len__(self) -> int:
        """
        Return number of windows in dataset.
        """
        return len(self.windows)

    def __getitem__(self, idx: int):
        """
        Load one training sample.

        Returns
        -------
        tuple
            (
                diff_frames,
                raw_frames,
                target_signal,
                validity_mask,
            )
        """
        (
            video_path,
            gt_path,
            frame_indices,
            total_frames,
            start,
        ) = self.windows[idx]

        # Load processed video streams
        (
            diff_frames,
            raw_frames,
            frame_indices,
            total_frames,
            n_valid,
        ) = process_video(
            video_path,
            use_cache=self.use_cache,
        )

        n_frames = n_valid - 1

        # -------------------------------------------------------------
        # Window slicing
        # -------------------------------------------------------------
        end = min(
            start + cfg.SEQ_LEN,
            n_frames,
        )

        diff_slice = diff_frames[start:end]
        raw_slice = raw_frames[start:end]

        # -------------------------------------------------------------
        # Load GT signal with exact frame alignment
        # -------------------------------------------------------------
        signal_full = load_ubfc_gt_indexed(
            gt_path,
            frame_indices,
            total_frames,
            cfg.FPS,
        )

        signal_slice = signal_full[start:end]

        # -------------------------------------------------------------
        # Data augmentation
        # -------------------------------------------------------------
        diff_slice, signal_slice = augment_sequence(
            diff_slice,
            signal_slice,
            self.training,
        )

        # Ensure raw stream matches augmented length
        raw_slice = raw_slice[: len(diff_slice)]

        sequence_length = len(signal_slice)

        # -------------------------------------------------------------
        # Pad or crop to fixed sequence length
        # -------------------------------------------------------------
        diff_out = pad_or_crop(
            diff_slice,
            cfg.SEQ_LEN,
            axis=0,
        )

        raw_out = pad_or_crop(
            raw_slice,
            cfg.SEQ_LEN,
            axis=0,
        )

        signal_out = pad_or_crop(
            signal_slice,
            cfg.SEQ_LEN,
            axis=0,
        )

        # -------------------------------------------------------------
        # Validity mask
        # 1.0 = real frame
        # 0.0 = padded frame
        # -------------------------------------------------------------
        mask = np.zeros(
            cfg.SEQ_LEN,
            dtype=np.float32,
        )

        valid_length = min(
            sequence_length,
            cfg.SEQ_LEN,
        )

        mask[:valid_length] = 1.0

        # -------------------------------------------------------------
        # Convert to tensors
        # -------------------------------------------------------------
        return (
            torch.tensor(
                diff_out,
                dtype=torch.float32,
            ),

            torch.tensor(
                raw_out,
                dtype=torch.float32,
            ),

            torch.tensor(
                signal_out,
                dtype=torch.float32,
            ),

            torch.tensor(
                mask,
                dtype=torch.float32,
            ),
        )


print("RPPGDataset v5 initialized successfully.")

RPPGDataset v5 initialized successfully.


In [20]:
import math
import torch
import torch.nn as nn

from torch.utils.checkpoint import checkpoint


# ──────────────────────────────────────────────────────────────────────────
# Squeeze-and-Excitation Block
# ──────────────────────────────────────────────────────────────────────────

class SEBlock(nn.Module):
    """
    Channel attention using squeeze-and-excitation.

    Enhances informative channels while suppressing
    less useful feature responses.
    """

    def __init__(
        self,
        channels: int,
        reduction: int = 4,
    ):
        super().__init__()

        reduced_channels = max(
            1,
            channels // reduction,
        )

        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),

            nn.Linear(
                channels,
                reduced_channels,
                bias=False,
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                reduced_channels,
                channels,
                bias=False,
            ),

            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        scale = self.se(x)

        scale = scale.unsqueeze(-1).unsqueeze(-1)

        return x * scale


# ──────────────────────────────────────────────────────────────────────────
# Spatial Attention
# ──────────────────────────────────────────────────────────────────────────

class SpatialAttention(nn.Module):
    """
    Spatial attention module.

    Learns spatial importance maps using
    channel-wise average and max pooling.
    """

    def __init__(self, kernel_size: int = 7):
        super().__init__()

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size,
            padding=kernel_size // 2,
            bias=False,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        avg_pool = x.mean(dim=1, keepdim=True)

        max_pool = x.max(dim=1, keepdim=True)[0]

        attention = torch.cat(
            [avg_pool, max_pool],
            dim=1,
        )

        attention = torch.sigmoid(
            self.conv(attention)
        )

        return x * attention


# ──────────────────────────────────────────────────────────────────────────
# CBAM Attention Block
# ──────────────────────────────────────────────────────────────────────────

class CBAMBlock(nn.Module):
    """
    Convolutional Block Attention Module (CBAM).

    Combines:
    - Channel attention
    - Spatial attention
    """

    def __init__(self, channels: int):
        super().__init__()

        self.channel_attention = SEBlock(channels)

        self.spatial_attention = SpatialAttention()

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        x = self.channel_attention(x)

        x = self.spatial_attention(x)

        return x


# ──────────────────────────────────────────────────────────────────────────
# Depthwise-Separable Convolution Block
# ──────────────────────────────────────────────────────────────────────────

def dw_sep(
    in_channels: int,
    out_channels: int,
    stride: int = 1,
) -> nn.Sequential:
    """
    Depthwise-separable convolution block.

    Reduces parameters and computation compared
    to standard convolution.
    """

    return nn.Sequential(

        # Depthwise convolution
        nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=in_channels,
            bias=False,
        ),

        nn.BatchNorm2d(in_channels),

        nn.ReLU(inplace=True),

        # Pointwise convolution
        nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False,
        ),

        nn.BatchNorm2d(out_channels),

        nn.ReLU(inplace=True),
    )


# ──────────────────────────────────────────────────────────────────────────
# Attention Pooling
# ──────────────────────────────────────────────────────────────────────────

class AttentionPool2d(nn.Module):
    """
    Spatial attention pooling.

    Produces weighted global feature vectors.
    """

    def __init__(self, channels: int):
        super().__init__()

        self.score = nn.Conv2d(
            channels,
            1,
            kernel_size=1,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        weights = self.score(x)

        weights = weights.flatten(2)

        weights = torch.softmax(
            weights,
            dim=-1,
        )

        features = x.flatten(2)

        pooled = (features * weights).sum(dim=-1)

        return pooled


# ──────────────────────────────────────────────────────────────────────────
# Spectral Attention
# ──────────────────────────────────────────────────────────────────────────

class SpectralAttn1D(nn.Module):
    """
    Frequency-domain attention.

    Uses FFT magnitude statistics to gate
    temporal transformer features.
    """

    def __init__(self, dim: int):
        super().__init__()

        hidden_dim = max(1, dim // 2)

        self.fc = nn.Sequential(

            nn.Linear(dim, hidden_dim),

            nn.ReLU(inplace=True),

            nn.Linear(hidden_dim, dim),

            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # FFT along temporal dimension
        spectrum = torch.fft.rfft(
            x,
            dim=1,
        )

        magnitude = torch.abs(spectrum)

        magnitude = magnitude.mean(dim=1)

        gate = self.fc(magnitude)

        gate = gate.unsqueeze(1)

        return x * gate


# ──────────────────────────────────────────────────────────────────────────
# Multi-Scale Spatial Encoder
# ──────────────────────────────────────────────────────────────────────────

class MultiScaleSpatialEncoder(nn.Module):
    """
    Multi-scale CNN encoder for spatial rPPG features.

    Extracts:
    - Shallow features
    - Mid-level features
    - Deep semantic features
    """

    def __init__(self, d_model: int = 64):
        super().__init__()

        def conv_bn(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
        ):
            return nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size,
                    stride,
                    padding,
                    bias=False,
                ),

                nn.BatchNorm2d(out_channels),

                nn.ReLU(inplace=True),
            )

        # -------------------------------------------------------------
        # Stem
        # -------------------------------------------------------------
        self.stem = nn.Sequential(
            conv_bn(3, 16),
            conv_bn(16, 16),
            nn.MaxPool2d(2),
        )

        # -------------------------------------------------------------
        # Scale 1
        # -------------------------------------------------------------
        self.scale1 = dw_sep(16, 16)

        self.cbam1 = CBAMBlock(16)

        self.pool1 = AttentionPool2d(16)

        # -------------------------------------------------------------
        # Scale 2
        # -------------------------------------------------------------
        self.scale2 = nn.Sequential(
            dw_sep(16, 32),
            dw_sep(32, 32),
            nn.MaxPool2d(2),
        )

        self.cbam2 = CBAMBlock(32)

        self.pool2 = AttentionPool2d(32)

        # -------------------------------------------------------------
        # Scale 3
        # -------------------------------------------------------------
        self.scale3 = nn.Sequential(
            dw_sep(32, 64),
            dw_sep(64, 64),
            nn.MaxPool2d(2),
        )

        self.cbam3 = CBAMBlock(64)

        self.pool3 = AttentionPool2d(64)

        # -------------------------------------------------------------
        # Projection Head
        # -------------------------------------------------------------
        total_features = 16 + 32 + 64

        self.proj = nn.Sequential(

            nn.Linear(
                total_features,
                d_model,
            ),

            nn.LayerNorm(d_model),

            nn.GELU(),

            nn.Dropout(cfg.DROPOUT),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        stem = self.stem(x)

        # Shallow features
        feat1 = self.scale1(stem)
        feat1 = self.cbam1(feat1)
        feat1 = self.pool1(feat1)

        # Mid-level features
        scale2 = self.scale2(stem)

        feat2 = self.cbam2(scale2)
        feat2 = self.pool2(feat2)

        # Deep features
        feat3 = self.scale3(scale2)
        feat3 = self.cbam3(feat3)
        feat3 = self.pool3(feat3)

        # Concatenate multi-scale features
        features = torch.cat(
            [feat1, feat2, feat3],
            dim=-1,
        )

        return self.proj(features)


# ──────────────────────────────────────────────────────────────────────────
# Sinusoidal Positional Encoding
# ──────────────────────────────────────────────────────────────────────────

class SinusoidalPE(nn.Module):
    """
    Standard sinusoidal positional encoding
    used in Transformers.
    """

    def __init__(
        self,
        dim: int,
        max_len: int = 2000,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, dim)

        position = torch.arange(
            max_len
        ).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, dim, 2).float()
            * (-math.log(10000.0) / dim)
        )

        pe[:, 0::2] = torch.sin(position * div_term)

        pe[:, 1::2] = torch.cos(
            position * div_term[: dim // 2]
        )

        self.register_buffer(
            "pe",
            pe.unsqueeze(0),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        x = x + self.pe[:, : x.size(1)]

        return self.dropout(x)


# ──────────────────────────────────────────────────────────────────────────
# Main rPPG Transformer
# ──────────────────────────────────────────────────────────────────────────

class RPPGTransformer(nn.Module):
    """
    Transformer-based rPPG estimation network.

    Inputs:
    -------
    diff : motion stream
    raw  : appearance stream

    Output:
    -------
    Predicted pulse waveform.
    """

    def __init__(self):
        super().__init__()

        # Motion encoder
        self.motion_encoder = MultiScaleSpatialEncoder(
            cfg.D_MODEL
        )

        # Appearance encoder
        self.appearance_encoder = MultiScaleSpatialEncoder(
            cfg.D_MODEL
        )

        # Feature fusion
        self.fuse = nn.Sequential(

            nn.Linear(
                cfg.D_MODEL * 2,
                cfg.D_MODEL,
            ),

            nn.LayerNorm(cfg.D_MODEL),

            nn.GELU(),
        )

        # Positional encoding
        self.pe = SinusoidalPE(
            cfg.D_MODEL,
            max_len=cfg.SEQ_LEN + 16,
            dropout=cfg.DROPOUT,
        )

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.D_MODEL,
            nhead=cfg.NHEAD,
            dim_feedforward=cfg.DIM_FF,
            dropout=cfg.DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=cfg.NUM_LAYERS,
            enable_nested_tensor=False,
        )

        # Spectral attention
        self.spec_attn = SpectralAttn1D(cfg.D_MODEL)

        # Temporal convolution refinement
        self.temp_conv = nn.Sequential(

            nn.Conv1d(
                cfg.D_MODEL,
                cfg.D_MODEL,
                kernel_size=5,
                padding=2,
                groups=cfg.D_MODEL,
                bias=False,
            ),

            nn.Conv1d(
                cfg.D_MODEL,
                cfg.D_MODEL,
                kernel_size=1,
                bias=False,
            ),

            nn.BatchNorm1d(cfg.D_MODEL),

            nn.GELU(),

            nn.Conv1d(
                cfg.D_MODEL,
                cfg.D_MODEL,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
        )

        # Prediction head
        self.head = nn.Sequential(

            nn.LayerNorm(cfg.D_MODEL),

            nn.Linear(cfg.D_MODEL, 1),
        )

        self._init_weights()

    def _init_weights(self):
        """
        Initialize model weights.
        """

        for module in self.modules():

            if isinstance(module, nn.Linear):

                nn.init.trunc_normal_(
                    module.weight,
                    std=0.02,
                )

                if module.bias is not None:
                    nn.init.zeros_(module.bias)

            elif isinstance(
                module,
                (
                    nn.BatchNorm1d,
                    nn.BatchNorm2d,
                    nn.LayerNorm,
                ),
            ):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def _encode_stream(
        self,
        encoder: nn.Module,
        x: torch.Tensor,
    ) -> torch.Tensor:
        """
        Encode frame sequences using batched CNN inference.

        Input shape:
            (B, T, H, W, C)

        Output shape:
            (B, T, D)
        """
        batch_size, time_steps, height, width, channels = x.shape

        x = x.permute(
            0,
            1,
            4,
            2,
            3,
        )

        x = x.reshape(
            batch_size * time_steps,
            channels,
            height,
            width,
        )

        if cfg.GRAD_CKPT and self.training:

            features = checkpoint(
                encoder,
                x,
                use_reentrant=False,
            )

        else:
            features = encoder(x)

        features = features.view(
            batch_size,
            time_steps,
            -1,
        )

        return features

    def forward(
        self,
        diff: torch.Tensor,
        raw: torch.Tensor,
        src_key_padding_mask=None,
    ) -> torch.Tensor:
        """
        Forward pass.

        Parameters
        ----------
        diff : torch.Tensor
            Motion stream.

        raw : torch.Tensor
            Appearance stream.

        src_key_padding_mask : torch.Tensor, optional
            Transformer padding mask.

        Returns
        -------
        torch.Tensor
            Predicted pulse waveform.
        """

        # Encode motion features
        motion_features = self._encode_stream(
            self.motion_encoder,
            diff,
        )

        # Encode appearance features
        appearance_features = self._encode_stream(
            self.appearance_encoder,
            raw,
        )

        # Feature fusion
        x = torch.cat(
            [motion_features, appearance_features],
            dim=-1,
        )

        x = self.fuse(x)

        # Add positional encoding
        x = self.pe(x)

        # Transformer modeling
        x = self.transformer(
            x,
            src_key_padding_mask=src_key_padding_mask,
        )

        # Spectral attention
        x = self.spec_attn(x)

        # Temporal convolution refinement
        residual = self.temp_conv(
            x.permute(0, 2, 1)
        )

        residual = residual.permute(0, 2, 1)

        x = x + residual

        # Final waveform prediction
        output = self.head(x)

        return output.squeeze(-1)


# -------------------------------------------------------------------------
# Model summary
# -------------------------------------------------------------------------

_model = RPPGTransformer()

num_params = sum(
    p.numel()
    for p in _model.parameters()
    if p.requires_grad
)

print(
    f"RPPGTransformer ready | "
    f"Trainable params: {num_params:,} "
    f"({num_params / 1e6:.2f}M)"
)

del _model

RPPGTransformer ready | Trainable params: 176,307 (0.18M)


## ⚖️ Loss Functions (v5 — All Bugs Fixed)

**FIX #3:** `negative_pearson` now computes mean only over valid (non-padded) frames.  
**FIX #6:** Spectral and SNR losses masked — zero-padding excluded from FFT.  
**FIX #11:** `snr_aware_loss` weight capped and curriculum-controlled.


In [21]:
import torch
import torch.nn.functional as F


# ──────────────────────────────────────────────────────────────────────────
# Negative Pearson Correlation Loss
# ──────────────────────────────────────────────────────────────────────────

def negative_pearson(
    preds: torch.Tensor,
    labels: torch.Tensor,
    mask: torch.Tensor = None,
) -> torch.Tensor:
    """
    Compute negative Pearson correlation loss.

    Important Fix
    -------------
    Mean statistics are computed ONLY over valid frames
    defined by the mask.

    This avoids correlation bias caused by zero-padded frames.

    Parameters
    ----------
    preds : torch.Tensor
        Predicted waveform tensor of shape (B, T).

    labels : torch.Tensor
        Ground-truth waveform tensor of shape (B, T).

    mask : torch.Tensor, optional
        Validity mask of shape (B, T).
        1 = valid frame
        0 = padded frame

    Returns
    -------
    torch.Tensor
        Mean negative Pearson correlation loss.
    """
    if mask is None:
        mask = torch.ones_like(preds)

    mask = mask.float()

    # Number of valid samples per sequence
    n_valid = mask.sum(
        dim=1,
        keepdim=True,
    ).clamp(min=1.0)

    # Masked means
    pred_mean = (
        (preds * mask).sum(dim=1, keepdim=True)
        / n_valid
    )

    label_mean = (
        (labels * mask).sum(dim=1, keepdim=True)
        / n_valid
    )

    # Center signals using masked means
    pred_centered = (preds - pred_mean) * mask

    label_centered = (labels - label_mean) * mask

    # Pearson numerator
    numerator = (
        pred_centered * label_centered
    ).sum(dim=1)

    # Pearson denominator
    denominator = torch.sqrt(
        (pred_centered.pow(2).sum(dim=1))
        * (label_centered.pow(2).sum(dim=1))
        + 1e-8
    )

    correlation = numerator / denominator

    return (1.0 - correlation).mean()


# ──────────────────────────────────────────────────────────────────────────
# Spectral Loss
# ──────────────────────────────────────────────────────────────────────────

def enhanced_spectral_loss(
    pred: torch.Tensor,
    target: torch.Tensor,
    mask: torch.Tensor,
    fs: float = 30.0,
) -> torch.Tensor:
    """
    Spectral-domain loss for waveform supervision.

    Important Fix
    -------------
    Padding mask is applied BEFORE FFT computation.

    This prevents:
    - Artificial DC components
    - Spectral leakage from padded frames
    - High-frequency FFT artifacts

    Components
    ----------
    1. L1 spectral magnitude loss
    2. KL divergence between normalized spectra

    Parameters
    ----------
    pred : torch.Tensor
        Predicted waveform tensor.

    target : torch.Tensor
        Ground-truth waveform tensor.

    mask : torch.Tensor
        Valid-frame mask.

    fs : float, default=30.0
        Sampling frequency in Hz.

    Returns
    -------
    torch.Tensor
        Spectral consistency loss.
    """
    batch_size, seq_len = pred.shape

    freqs = torch.fft.rfftfreq(
        seq_len,
        d=1.0 / fs,
    ).to(pred.device)

    # Physiological frequency band
    band_mask = (
        (freqs >= FREQ_LOW)
        & (freqs <= FREQ_HIGH)
    )

    # Apply mask before FFT
    pred_masked = pred * mask
    target_masked = target * mask

    # Hann window
    window = torch.hann_window(
        seq_len,
        device=pred.device,
    )

    # FFT magnitudes
    pred_fft = torch.fft.rfft(
        pred_masked * window,
        dim=1,
    )

    target_fft = torch.fft.rfft(
        target_masked * window,
        dim=1,
    )

    pred_mag = torch.abs(pred_fft)[:, band_mask]

    target_mag = torch.abs(target_fft)[:, band_mask]

    # L1 spectral distance
    l1_loss = torch.mean(
        torch.abs(pred_mag - target_mag)
    )

    # Normalize spectra
    pred_norm = pred_mag / (
        pred_mag.sum(dim=1, keepdim=True)
        + 1e-8
    )

    target_norm = target_mag / (
        target_mag.sum(dim=1, keepdim=True)
        + 1e-8
    )

    # KL divergence
    kl_loss = F.kl_div(
        torch.log(pred_norm + 1e-8),
        target_norm,
        reduction="batchmean",
    )

    return l1_loss + 0.1 * kl_loss


# ──────────────────────────────────────────────────────────────────────────
# SNR-Aware Loss
# ──────────────────────────────────────────────────────────────────────────

def snr_aware_loss(
    pred: torch.Tensor,
    mask: torch.Tensor,
    fs: float = 30.0,
) -> torch.Tensor:
    """
    Penalize out-of-band spectral energy.

    Important Fix
    -------------
    Padding mask is applied BEFORE FFT.

    This prevents padded regions from creating
    artificial spectral energy.

    Parameters
    ----------
    pred : torch.Tensor
        Predicted waveform tensor.

    mask : torch.Tensor
        Valid-frame mask.

    fs : float, default=30.0
        Sampling frequency in Hz.

    Returns
    -------
    torch.Tensor
        Mean out-of-band energy fraction.
    """
    seq_len = pred.shape[1]

    freqs = torch.fft.rfftfreq(
        seq_len,
        d=1.0 / fs,
    ).to(pred.device)

    band_mask = (
        (freqs >= FREQ_LOW)
        & (freqs <= FREQ_HIGH)
    )

    # Apply mask before FFT
    pred_masked = pred * mask

    power_spectrum = torch.abs(
        torch.fft.rfft(
            pred_masked,
            dim=1,
        )
    ) ** 2

    band_power = power_spectrum[
        :,
        band_mask,
    ].sum(dim=1)

    total_power = (
        power_spectrum.sum(dim=1)
        + 1e-8
    )

    out_of_band_fraction = (
        1.0 - band_power / total_power
    )

    return out_of_band_fraction.mean()


# ──────────────────────────────────────────────────────────────────────────
# Contrastive Temporal Loss
# ──────────────────────────────────────────────────────────────────────────

def contrastive_temporal_loss(
    pred: torch.Tensor,
    temperature: float = 0.07,
) -> torch.Tensor:
    """
    Contrastive spectral consistency loss.

    Splits each temporal sequence into two halves
    and enforces spectral similarity.

    Important Fix
    -------------
    Handles small batch sizes safely.

    Parameters
    ----------
    pred : torch.Tensor
        Predicted waveform tensor.

    temperature : float
        Contrastive temperature scaling.

    Returns
    -------
    torch.Tensor
        Contrastive loss value.
    """
    batch_size, seq_len = pred.shape

    # Guard for small sequences or batch size
    if seq_len < 32 or batch_size < 2:

        return torch.tensor(
            0.0,
            device=pred.device,
        )

    half = seq_len // 2

    # First half FFT
    z1 = torch.abs(
        torch.fft.rfft(
            pred[:, :half],
            dim=1,
        )
    )

    # Second half FFT
    z2 = torch.abs(
        torch.fft.rfft(
            pred[:, half:2 * half],
            dim=1,
        )
    )

    # Normalize embeddings
    z1 = F.normalize(z1, dim=1)

    z2 = F.normalize(z2, dim=1)

    # Similarity logits
    logits = torch.mm(
        z1,
        z2.T,
    ) / temperature

    labels = torch.arange(
        batch_size,
        device=pred.device,
    )

    # Bidirectional contrastive loss
    loss_forward = F.cross_entropy(
        logits,
        labels,
    )

    loss_backward = F.cross_entropy(
        logits.T,
        labels,
    )

    return 0.5 * (
        loss_forward + loss_backward
    )


# ──────────────────────────────────────────────────────────────────────────
# Total Multi-Component Loss
# ──────────────────────────────────────────────────────────────────────────

def total_loss(
    pred: torch.Tensor,
    target: torch.Tensor,
    mask: torch.Tensor,
    cfg,
    epoch: int = 0,
    fs: float = 30.0,
) -> tuple:
    """
    Combined training loss for rPPG waveform learning.

    Includes:
    ---------
    - Pearson correlation loss
    - Smooth L1 waveform loss
    - Spectral consistency loss
    - SNR-aware frequency regularization
    - Contrastive temporal consistency

    Features
    --------
    - Curriculum learning:
      Spectral losses activate only after warmup.

    - NaN/Inf safety guard.

    Parameters
    ----------
    pred : torch.Tensor
        Predicted waveform tensor.

    target : torch.Tensor
        Ground-truth waveform tensor.

    mask : torch.Tensor
        Valid-frame mask.

    cfg : object
        Training configuration object.

    epoch : int, default=0
        Current training epoch.

    fs : float, default=30.0
        Sampling frequency.

    Returns
    -------
    tuple
        (
            total_loss,
            component_dictionary,
        )
    """
    components = {}

    # -------------------------------------------------------------
    # Core losses
    # -------------------------------------------------------------
    components["pearson"] = negative_pearson(
        pred,
        target,
        mask,
    )

    components["smooth_l1"] = F.smooth_l1_loss(
        pred * mask,
        target * mask,
    )

    # -------------------------------------------------------------
    # Curriculum learning
    # -------------------------------------------------------------
    past_warmup = (
        epoch >= cfg.WARMUP_EPOCHS
    )

    if past_warmup:

        components["spectral"] = enhanced_spectral_loss(
            pred,
            target,
            mask,
            fs,
        )

        components["snr"] = snr_aware_loss(
            pred,
            mask,
            fs,
        )

    else:

        components["spectral"] = torch.tensor(
            0.0,
            device=pred.device,
        )

        components["snr"] = torch.tensor(
            0.0,
            device=pred.device,
        )

    # Contrastive loss
    components["contrast"] = (
        contrastive_temporal_loss(pred)
    )

    # -------------------------------------------------------------
    # Weighted total loss
    # -------------------------------------------------------------
    total = (

        cfg.W_PEARSON
        * components["pearson"]

        + cfg.W_SMOOTH
        * components["smooth_l1"]

        + cfg.W_SPECTRAL
        * components["spectral"]

        + cfg.W_SNR
        * components["snr"]

        + cfg.W_CONTRAST
        * components["contrast"]
    )

    # -------------------------------------------------------------
    # Numerical stability guard
    # -------------------------------------------------------------
    if not torch.isfinite(total):

        print(
            "Warning: NaN/Inf detected in loss. "
            "Returning zero loss."
        )

        total = torch.tensor(
            0.0,
            device=pred.device,
            requires_grad=True,
        )

    return total, components


# ──────────────────────────────────────────────────────────────────────────
# Initialization Message
# ──────────────────────────────────────────────────────────────────────────

print("Loss functions v5 initialized successfully.")

print("Fixes enabled:")
print("  - Masked Pearson correlation")
print("  - FFT masking before spectral analysis")
print("  - Curriculum-gated spectral supervision")
print("  - Stable SNR regularization")
print("  - Batch-safe contrastive learning")

Loss functions v5 initialized successfully.
Fixes enabled:
  - Masked Pearson correlation
  - FFT masking before spectral analysis
  - Curriculum-gated spectral supervision
  - Stable SNR regularization
  - Batch-safe contrastive learning


In [22]:
import numpy as np

from scipy.stats import pearsonr


# ──────────────────────────────────────────────────────────────────────────
# Evaluation Metrics
# ──────────────────────────────────────────────────────────────────────────

def compute_metrics(
    pred_signals,
    gt_signals,
    fs: float = 30.0,
) -> dict:
    """
    Compute evaluation metrics for predicted rPPG signals.

    Metrics
    -------
    - MAE (BPM)
    - RMSE (BPM)
    - Pearson correlation
    - Average SNR (dB)

    Parameters
    ----------
    pred_signals : list or np.ndarray
        List of predicted waveform signals.

    gt_signals : list or np.ndarray
        List of ground-truth waveform signals.

    fs : float, default=30.0
        Sampling frequency in Hz.

    Returns
    -------
    dict
        Dictionary containing evaluation metrics.
    """
    pred_bpms = []
    gt_bpms = []

    pearson_scores = []
    snr_scores = []

    for pred_signal, gt_signal in zip(pred_signals, gt_signals):

        pred_signal = np.asarray(
            pred_signal,
            dtype=np.float32,
        )

        gt_signal = np.asarray(
            gt_signal,
            dtype=np.float32,
        )

        # -------------------------------------------------------------
        # BPM estimation
        # -------------------------------------------------------------
        pred_bpm = compute_bpm(
            pred_signal,
            fs,
        )

        gt_bpm = compute_bpm(
            gt_signal,
            fs,
        )

        pred_bpms.append(pred_bpm)
        gt_bpms.append(gt_bpm)

        # -------------------------------------------------------------
        # Pearson correlation
        # -------------------------------------------------------------
        try:
            corr, _ = pearsonr(
                pred_signal,
                gt_signal,
            )

            if np.isfinite(corr):
                pearson_scores.append(float(corr))
            else:
                pearson_scores.append(0.0)

        except Exception:
            pearson_scores.append(0.0)

        # -------------------------------------------------------------
        # Signal-to-noise ratio
        # -------------------------------------------------------------
        snr_scores.append(
            compute_snr(
                pred_signal,
                fs,
            )
        )

    # Convert to arrays
    pred_bpms = np.asarray(pred_bpms)
    gt_bpms = np.asarray(gt_bpms)

    # Absolute BPM errors
    bpm_errors = np.abs(
        pred_bpms - gt_bpms
    )

    # Root mean squared error
    rmse = np.sqrt(
        np.mean(bpm_errors ** 2)
    )

    return {

        "mae_bpm": float(
            np.mean(bpm_errors)
        ),

        "rmse_bpm": float(rmse),

        "pearson_r": float(
            np.mean(pearson_scores)
        ),

        "snr_db": float(
            np.mean(snr_scores)
        ),

        # Raw arrays for deeper analysis
        "pred_bpms": pred_bpms,

        "gt_bpms": gt_bpms,

        "errors": bpm_errors,
    }


# ──────────────────────────────────────────────────────────────────────────
# Prediction Collapse Detector
# ──────────────────────────────────────────────────────────────────────────

def check_prediction_collapse(
    preds: np.ndarray,
    threshold_std: float = 0.05,
) -> bool:
    """
    Detect degenerate prediction collapse.

    A model is considered collapsed if predicted BPM values
    have extremely low variance across samples.

    Parameters
    ----------
    preds : np.ndarray
        Predicted waveform signals.

    threshold_std : float, default=0.05
        Collapse threshold expressed as a fraction
        of the physiological BPM range.

    Returns
    -------
    bool
        True if collapse is detected.
    """
    if len(preds) == 0:
        return True

    # -------------------------------------------------------------
    # Compute BPM for each prediction
    # -------------------------------------------------------------
    bpm_values = np.asarray(
        [compute_bpm(p) for p in preds],
        dtype=np.float32,
    )

    bpm_std = bpm_values.std()

    # Threshold relative to physiological BPM range
    collapse_threshold = (
        threshold_std
        * (BPM_HIGH - BPM_LOW)
    )

    is_collapsed = (
        bpm_std < collapse_threshold
    )

    # -------------------------------------------------------------
    # Diagnostic output
    # -------------------------------------------------------------
    if is_collapsed:

        signal_std = np.stack(preds).std(
            axis=1
        ).mean()

        print(
            "\n⚠️ Prediction collapse detected"
        )

        print(
            f"  Predicted BPMs: "
            f"{bpm_values.mean():.1f} ± {bpm_std:.2f}"
        )

        print(
            f"  Threshold: {collapse_threshold:.2f} BPM"
        )

        print(
            f"  Mean signal std: {signal_std:.6f}"
        )

        print("\nPossible causes:")

        print(
            "  - Incorrect GT alignment"
        )

        print(
            "  - Learning rate too high"
        )

        print(
            "  - Invalid bandpass output"
        )

        print(
            "  - Spectral loss overpowering waveform loss"
        )

        print(
            "  - Data normalization issues"
        )

    return is_collapsed


# ──────────────────────────────────────────────────────────────────────────
# Initialization Message
# ──────────────────────────────────────────────────────────────────────────

collapse_limit = (
    0.05 * (BPM_HIGH - BPM_LOW)
)

print("Metrics and collapse detector initialized.")

print(
    f"Physiological BPM range: "
    f"[{BPM_LOW:.1f}, {BPM_HIGH:.1f}]"
)

print(
    f"Collapse threshold: "
    f"std < {collapse_limit:.2f} BPM"
)

Metrics and collapse detector initialized.
Physiological BPM range: [45.0, 150.0]
Collapse threshold: std < 5.25 BPM


## 🏋️ Training Setup (v5)

**FIX #4 (Warmup):** `WARMUP_EPOCHS = max(1, EPOCHS//10)` → only 3 warmup epochs for 50-epoch CPU run.  
**FIX #9 (EMA):** `EMA_DECAY = 0.99` on CPU for faster EMA convergence.  
Loss weights rebalanced; spectral/SNR gated by curriculum.


In [24]:
# set_seed(42)
# model = RPPGTransformer().to(cfg.DEVICE)
# n = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"Model: RPPGTransformer  Params: {n:,} ({n/1e6:.2f}M)")

# # ── Optimiser: separate LRs for CNN vs Transformer ──────────────────────
# param_groups = [
#     {'params': list(model.motion_enc.parameters()) +
#                list(model.appearance_enc.parameters()),
#      'lr': cfg.LR * 0.5, 'weight_decay': cfg.WEIGHT_DECAY},
#     {'params': list(model.fuse.parameters()) +
#                list(model.transformer.parameters()) +
#                list(model.pe.parameters()) +
#                list(model.spec_attn.parameters()) +
#                list(model.temp_conv.parameters()) +
#                list(model.head.parameters()),
#      'lr': cfg.LR, 'weight_decay': cfg.WEIGHT_DECAY * 0.1},
# ]
# optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.95))

# # ── FIX #4: Warmup limited to EPOCHS//10 so cosine phase is reached ─────
# def warmup_cosine_lambda(epoch: int) -> float:
#     if epoch < cfg.WARMUP_EPOCHS:
#         return (epoch + 1) / cfg.WARMUP_EPOCHS
#     progress = (epoch - cfg.WARMUP_EPOCHS) / max(1, cfg.EPOCHS - cfg.WARMUP_EPOCHS)
#     return 0.5 * (1.0 + math.cos(math.pi * progress))

# scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, warmup_cosine_lambda)

# # ── FIX #9: EMA with adaptive decay ─────────────────────────────────────
# class EMAModel:
#     def __init__(self, model, decay=0.999):
#         self.decay  = decay
#         self.shadow = {k: v.clone().float() for k, v in model.state_dict().items()}
#     @torch.no_grad()
#     def update(self, model):
#         for k, v in model.state_dict().items():
#             self.shadow[k].mul_(self.decay).add_(v.float(), alpha=1-self.decay)
#     def apply(self, model):
#         model.load_state_dict(
#             {k: v.to(next(model.parameters()).device) for k,v in self.shadow.items()}
#         )
#     def restore(self, original_sd, model):
#         model.load_state_dict(original_sd)

# ema = EMAModel(model, decay=cfg.EMA_DECAY)   # FIX #9: 0.99 on CPU

# # ── AMP Scaler ───────────────────────────────────────────────────────────
# scaler = GradScaler(enabled=cfg.USE_AMP)

# # ── CSV logger ───────────────────────────────────────────────────────────
# log_path = Path(cfg.CKPT_DIR) / cfg.LOG_FILE
# with open(log_path, 'w', newline='') as f:
#     csv.writer(f).writerow([
#         'epoch','train_loss','val_loss','mae_bpm','rmse_bpm',
#         'pearson_r','snr_db','lr','collapse'
#     ])

# print(f"Optimiser  : AdamW  CNN_LR={cfg.LR*0.5:.1e}  TF_LR={cfg.LR:.1e}")
# print(f"Scheduler  : Warmup {cfg.WARMUP_EPOCHS}ep → CosineAnnealing")
# print(f"EMA decay  : {cfg.EMA_DECAY}  AMP: {cfg.USE_AMP}")
# print(f"Grad clip  : {cfg.GRAD_CLIP}  Accum: {cfg.ACCUM_STEPS} steps")
# print()
# # Print LR schedule for first 10 epochs
# print("LR schedule preview:")
# for ep in range(min(10, cfg.EPOCHS)):
#     mult = warmup_cosine_lambda(ep)
#     tf_lr = mult * cfg.LR
#     print(f"  Epoch {ep+1:3d}: mult={mult:.3f}  TF_LR={tf_lr:.2e}"
#           + (" ← warmup" if ep < cfg.WARMUP_EPOCHS else " ← cosine"))








set_seed(42)

model = RPPGTransformer().to(cfg.DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: RPPGTransformer | Params: {n_params:,} ({n_params/1e6:.2f}M)")


# ──────────────────────────────────────────────────────────────────────────
# Optimizer (separate learning rates)
# ──────────────────────────────────────────────────────────────────────────

cnn_params = (
    list(model.motion_encoder.parameters()) +
    list(model.appearance_encoder.parameters())
)

transformer_params = (
    list(model.fuse.parameters()) +
    list(model.transformer.parameters()) +
    list(model.pe.parameters()) +
    list(model.spec_attn.parameters()) +
    list(model.temp_conv.parameters()) +
    list(model.head.parameters())
)

param_groups = [
    {
        "params": cnn_params,
        "lr": cfg.LR * 0.5,
        "weight_decay": cfg.WEIGHT_DECAY,
    },
    {
        "params": transformer_params,
        "lr": cfg.LR,
        "weight_decay": cfg.WEIGHT_DECAY * 0.1,
    },
]

optimizer = torch.optim.AdamW(
    param_groups,
    betas=(0.9, 0.95),
)


# ──────────────────────────────────────────────────────────────────────────
# Warmup + Cosine Scheduler
# ──────────────────────────────────────────────────────────────────────────

def warmup_cosine_lambda(epoch: int) -> float:
    if epoch < cfg.WARMUP_EPOCHS:
        return (epoch + 1) / max(1, cfg.WARMUP_EPOCHS)

    progress = (
        (epoch - cfg.WARMUP_EPOCHS)
        / max(1, cfg.EPOCHS - cfg.WARMUP_EPOCHS)
    )

    return 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=warmup_cosine_lambda,
)


# ──────────────────────────────────────────────────────────────────────────
# EMA Model (fixed + safer)
# ──────────────────────────────────────────────────────────────────────────

class EMAModel:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.device = next(model.parameters()).device

        self.shadow = {
            k: v.detach().clone().float().to(self.device)
            for k, v in model.state_dict().items()
        }

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            v = v.detach().float().to(self.device)
            self.shadow[k].mul_(self.decay).add_(v, alpha=1.0 - self.decay)

    def apply(self, model):
        model.load_state_dict(
            {k: v.to(self.device) for k, v in self.shadow.items()}
        )

    def restore(self, model_state_dict, model):
        model.load_state_dict(model_state_dict)


ema = EMAModel(model, decay=cfg.EMA_DECAY)


# ──────────────────────────────────────────────────────────────────────────
# AMP Scaler (safe API)
# ──────────────────────────────────────────────────────────────────────────

scaler = torch.cuda.amp.GradScaler(
    enabled=cfg.USE_AMP
)


# ──────────────────────────────────────────────────────────────────────────
# Logging
# ──────────────────────────────────────────────────────────────────────────

log_path = Path(cfg.CKPT_DIR) / cfg.LOG_FILE

with open(log_path, "w", newline="") as f:
    csv.writer(f).writerow([
        "epoch",
        "train_loss",
        "val_loss",
        "mae_bpm",
        "rmse_bpm",
        "pearson_r",
        "snr_db",
        "lr",
        "collapse",
    ])


print(
    f"Optimizer  : AdamW | CNN LR={cfg.LR*0.5:.2e} | TF LR={cfg.LR:.2e}"
)
print(
    f"Scheduler  : Warmup {cfg.WARMUP_EPOCHS} → Cosine decay"
)
print(
    f"EMA decay  : {cfg.EMA_DECAY} | AMP: {cfg.USE_AMP}"
)
print(
    f"Grad clip  : {cfg.GRAD_CLIP} | Accum: {cfg.ACCUM_STEPS}"
)


# ──────────────────────────────────────────────────────────────────────────
# LR preview
# ──────────────────────────────────────────────────────────────────────────

print("\nLR schedule preview:")

for ep in range(min(10, cfg.EPOCHS)):

    mult = warmup_cosine_lambda(ep)

    lr_tf = mult * cfg.LR

    phase = (
        "warmup"
        if ep < cfg.WARMUP_EPOCHS
        else "cosine"
    )

    print(
        f"  Epoch {ep+1:3d}: "
        f"mult={mult:.3f} | LR={lr_tf:.2e} | {phase}"
    )

Model: RPPGTransformer | Params: 176,307 (0.18M)
Optimizer  : AdamW | CNN LR=1.50e-04 | TF LR=3.00e-04
Scheduler  : Warmup 3 → Cosine decay
EMA decay  : 0.99 | AMP: False
Grad clip  : 1.0 | Accum: 4

LR schedule preview:
  Epoch   1: mult=0.333 | LR=1.00e-04 | warmup
  Epoch   2: mult=0.667 | LR=2.00e-04 | warmup
  Epoch   3: mult=1.000 | LR=3.00e-04 | warmup
  Epoch   4: mult=1.000 | LR=3.00e-04 | cosine
  Epoch   5: mult=0.999 | LR=3.00e-04 | cosine
  Epoch   6: mult=0.996 | LR=2.99e-04 | cosine
  Epoch   7: mult=0.990 | LR=2.97e-04 | cosine
  Epoch   8: mult=0.982 | LR=2.95e-04 | cosine
  Epoch   9: mult=0.972 | LR=2.92e-04 | cosine
  Epoch  10: mult=0.960 | LR=2.88e-04 | cosine


## 🔄 Training & Validation Loops (v5)

All loop-level fixes:
- Epoch number passed to `total_loss` for curriculum gating (FIX #11)
- EMA restore uses `restore()` helper
- Collapse detection in validation


In [26]:
def train_one_epoch(loader: DataLoader, epoch: int) -> dict:
    model.train()
    running = {k: 0.0 for k in ['total','pearson','smooth_l1','spectral','snr','contrast']}
    optimizer.zero_grad()

    for step, (diff, raw, sig, mask) in enumerate(
        tqdm(loader, desc=f"Train E{epoch+1:3d}", leave=False)
    ):
        diff = diff.to(cfg.DEVICE, non_blocking=True)
        raw  = raw.to( cfg.DEVICE, non_blocking=True)
        sig  = sig.to( cfg.DEVICE, non_blocking=True)
        mask = mask.to(cfg.DEVICE, non_blocking=True)
        pad_mask = (mask == 0)

        with autocast(enabled=cfg.USE_AMP):
            pred         = model(diff, raw, src_key_padding_mask=pad_mask)
            loss, comps  = total_loss(pred, sig, mask, cfg, epoch=epoch, fs=cfg.FPS)
            loss_scaled  = loss / cfg.ACCUM_STEPS

        if not torch.isfinite(loss):
            print(f"  ⚠️ NaN step {step} — skip")
            optimizer.zero_grad()
            continue

        scaler.scale(loss_scaled).backward()

        if (step+1) % cfg.ACCUM_STEPS == 0 or (step+1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            ema.update(model)

        running['total'] += loss.item()
        for k, v in comps.items():
            if torch.is_tensor(v):
                running[k] = running.get(k, 0.0) + v.item()

    n = max(len(loader), 1)
    return {k: v/n for k,v in running.items()}


@torch.no_grad()
def validate(loader: DataLoader, epoch: int) -> dict:
    # "\"\"Validate using EMA weights; restore training weights after.\"\"\"
    # Swap in EMA weights
    original_sd = {k: v.clone() for k, v in model.state_dict().items()}
    ema.apply(model)
    model.eval()

    total_loss_val = 0.0
    all_preds, all_gts = [], []

    for diff, raw, sig, mask in tqdm(loader, desc="Val      ", leave=False):
        diff = diff.to(cfg.DEVICE, non_blocking=True)
        raw  = raw.to( cfg.DEVICE, non_blocking=True)
        sig  = sig.to( cfg.DEVICE, non_blocking=True)
        mask = mask.to(cfg.DEVICE, non_blocking=True)
        pad_mask = (mask == 0)

        with autocast(enabled=cfg.USE_AMP):
            pred    = model(diff, raw, src_key_padding_mask=pad_mask)
            loss, _ = total_loss(pred, sig, mask, cfg, epoch=epoch, fs=cfg.FPS)

        total_loss_val += loss.item()

        for b in range(pred.shape[0]):
            valid_len = int(mask[b].sum().item())
            if valid_len > 16:
                all_preds.append(pred[b, :valid_len].cpu().numpy())
                all_gts.append(sig[b,  :valid_len].cpu().numpy())

    # Restore training weights
    ema.restore(original_sd, model)

    if len(all_preds) == 0:
        return {'val_loss': total_loss_val/max(len(loader),1),
                'mae_bpm':999.,'rmse_bpm':999.,'pearson_r':0.,'snr_db':-99.,'collapse':True}

    min_len   = min(len(p) for p in all_preds)
    preds_arr = np.stack([p[:min_len] for p in all_preds])
    gts_arr   = np.stack([g[:min_len] for g in all_gts])

    metrics   = compute_metrics(preds_arr, gts_arr, cfg.FPS)
    metrics['val_loss'] = total_loss_val / max(len(loader), 1)
    metrics['collapse'] = check_prediction_collapse(all_preds)
    return metrics


print("Training and validation loops v5 ready.")


Training and validation loops v5 ready.


In [27]:
import glob

# ── Discover all vid.avi files ────────────────────────────────────────────
vid_pattern  = os.path.join(cfg.UBFC_ROOT, "*", "vid.avi")
all_vid      = sorted(glob.glob(vid_pattern))
if not all_vid:
    all_vid  = sorted(glob.glob(os.path.join(cfg.UBFC_ROOT, "*", "*.mp4")))
if not all_vid:
    all_vid  = sorted(glob.glob(os.path.join(cfg.UBFC_ROOT, "*", "*.avi")))

if not all_vid:
    raise FileNotFoundError(
        f"\\n❌  No videos found under: {cfg.UBFC_ROOT}\\n"
        f"    Expected: <UBFC_ROOT>/subjectN/vid.avi\\n"
        f"    Searched: {vid_pattern}"
    )

# ── Pair with GT ──────────────────────────────────────────────────────────
pairs = []
for vp in all_vid:
    gp = os.path.join(os.path.dirname(vp), cfg.GT_FILENAME)
    if os.path.isfile(gp):
        pairs.append((vp, gp))
    else:
        print(f"  ⚠️  No GT for {Path(vp).parent.name} — skipped")

all_vid = [p[0] for p in pairs]
all_gt  = [p[1] for p in pairs]
n_total = len(all_vid)

if n_total == 0:
    raise RuntimeError("No valid subject pairs found. Check UBFC_ROOT and GT_FILENAME.")

# ── Reproducible shuffle + 80/20 split ───────────────────────────────────
idx = list(range(n_total))
random.seed(42)
random.shuffle(idx)
all_vid = [all_vid[i] for i in idx]
all_gt  = [all_gt[i]  for i in idx]

n_train = max(1, int(cfg.TRAIN_RATIO * n_total))
n_val   = n_total - n_train

train_vp = all_vid[:n_train];  train_gp = all_gt[:n_train]
val_vp   = all_vid[n_train:];  val_gp   = all_gt[n_train:]

print(f"✅  Found {n_total} subjects with valid GT.")
print(f"    Train: {n_train} subjects  |  Val: {n_val} subjects")
print()
print("First 3 training subjects:")
for vp, gp in zip(train_vp[:3], train_gp[:3]):
    print(f"  {Path(vp).parent.name}  |  video: {Path(vp).name}  |  gt: {Path(gp).name}")

# ── Window count estimate ─────────────────────────────────────────────────
# UBFC videos ≈ 60s at 30fps = 1800 frames → ~1799 after diff
# Windows = (1799 - SEQ_LEN) / STRIDE + 1
est_win = max(1, (1799 - cfg.SEQ_LEN) // cfg.STRIDE + 1)
print(f"\\nEstimated windows per video: ~{est_win}")
print(f"Estimated total train windows: ~{n_train * est_win}")
print(f"(Actual count computed when building RPPGDataset)")


✅  Found 42 subjects with valid GT.
    Train: 33 subjects  |  Val: 9 subjects

First 3 training subjects:
  subject1  |  video: vid.avi  |  gt: ground_truth.txt
  subject13  |  video: vid.avi  |  gt: ground_truth.txt
  subject20  |  video: vid.avi  |  gt: ground_truth.txt
\nEstimated windows per video: ~53
Estimated total train windows: ~1749
(Actual count computed when building RPPGDataset)


In [28]:
train_dataset = RPPGDataset(train_vp, train_gp, training=True,  use_cache=cfg.USE_CACHE)
val_dataset   = RPPGDataset(val_vp,   val_gp,   training=False, use_cache=cfg.USE_CACHE)

_pin = (cfg.DEVICE == "cuda")

train_loader = DataLoader(
    train_dataset, batch_size=cfg.BATCH_SIZE,
    shuffle=True, num_workers=cfg.NUM_WORKERS,
    worker_init_fn=worker_init_fn, pin_memory=_pin,
    drop_last=True,
    persistent_workers=(cfg.NUM_WORKERS > 0),
    prefetch_factor=(2 if cfg.NUM_WORKERS > 0 else None),
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.BATCH_SIZE,
    shuffle=False, num_workers=cfg.NUM_WORKERS,
    worker_init_fn=worker_init_fn, pin_memory=_pin,
    drop_last=False,
    persistent_workers=(cfg.NUM_WORKERS > 0),
    prefetch_factor=(2 if cfg.NUM_WORKERS > 0 else None),
)

print(f"Train: {len(train_loader):4d} batches  ({len(train_dataset)} windows)")
print(f"Val:   {len(val_loader):4d} batches  ({len(val_dataset)} windows)")
print(f"Device: {cfg.DEVICE.upper()}  pin_memory: {_pin}")
if cfg.DEVICE == 'cpu':
    # Very rough timing estimate
    est_s = len(train_loader) * 1.5   # ~1.5s/batch on CPU with cache
    print(f"Rough CPU estimate: ~{est_s:.0f}s/epoch  (~{est_s*cfg.EPOCHS/3600:.1f}h total)")
    print("Tip: first epoch is slower (video processing + caching). Subsequent epochs use cache.")


Building train dataset...
  -> 33 videos | 1893 windows (SEQ_LEN=128, STRIDE=32)
Building val dataset...
  -> 9 videos | 506 windows (SEQ_LEN=128, STRIDE=32)
Train: 1893 batches  (1893 windows)
Val:    506 batches  (506 windows)
Device: CPU  pin_memory: False
Rough CPU estimate: ~2840s/epoch  (~39.4h total)
Tip: first epoch is slower (video processing + caching). Subsequent epochs use cache.


## 🔬 Pre-Training Sanity Check
Run this BEFORE training to verify:
1. GT signals are valid (non-zero, correct BPM range)
2. Frame–signal alignment looks correct
3. Dataloader produces correct tensor shapes


In [31]:
# print("=" * 60)
# print("  PRE-TRAINING SANITY CHECKS")
# print("=" * 60)

# # ── 1. Load one batch and check shapes ──────────────────────────────────
# diff_b, raw_b, sig_b, mask_b = next(iter(train_loader))
# print(f"\\n1. Tensor shapes from DataLoader:")
# print(f"   diff_frames : {tuple(diff_b.shape)}  (expect: B={cfg.BATCH_SIZE}, T={cfg.SEQ_LEN}, H={cfg.IMG_SIZE}, W={cfg.IMG_SIZE}, C=3)")
# print(f"   raw_frames  : {tuple(raw_b.shape)}")
# print(f"   signal      : {tuple(sig_b.shape)}  (expect: B={cfg.BATCH_SIZE}, T={cfg.SEQ_LEN})")
# print(f"   mask        : {tuple(mask_b.shape)}")
# print(f"   diff range  : [{diff_b.min():.3f}, {diff_b.max():.3f}]")
# print(f"   signal std  : {sig_b.std():.3f}  (expect ~0.9–1.1 after z-score)")
# print(f"   mask sum    : {mask_b.sum(dim=1).float().mean():.0f} valid frames avg")

# # ── 2. Check GT BPM distribution ────────────────────────────────────────
# print(f"\\n2. GT BPM distribution (from first 20 train windows):")
# bpm_vals = []
# for i in range(min(20, len(train_dataset))):
#     _, _, sig, msk = train_dataset[i]
#     vl = int(msk.sum().item())
#     if vl > 32:
#         bpm_vals.append(compute_bpm(sig[:vl].numpy()))
# if bpm_vals:
#     bpm_arr = np.array(bpm_vals)
#     print(f"   BPM range   : [{bpm_arr.min():.1f}, {bpm_arr.max():.1f}] BPM")
#     print(f"   BPM mean    : {bpm_arr.mean():.1f} BPM")
#     print(f"   BPM std     : {bpm_arr.std():.1f} BPM")
#     if bpm_arr.std() < 3.0:
#         print("   ⚠️  BPM std very low — possible GT alignment issue!")
#     else:
#         print("   ✅  BPM distribution looks healthy (std > 3 BPM)")
# else:
#     print("   ⚠️  No valid signals found — check dataset!")

# # ── 3. Model forward pass check ──────────────────────────────────────────
# print(f"\\n3. Model forward pass:")
# model.eval()
# with torch.no_grad():
#     out = model(diff_b[:1].to(cfg.DEVICE), raw_b[:1].to(cfg.DEVICE))
# print(f"   Output shape : {tuple(out.shape)}  (expect: 1, {cfg.SEQ_LEN})")
# print(f"   Output std   : {out.std().item():.4f}  (should be > 0 — if ~0 → collapsed output)")
# print(f"   Output range : [{out.min().item():.3f}, {out.max().item():.3f}]")
# model.train()

# # ── 4. Loss sanity ───────────────────────────────────────────────────────
# print(f"\\n4. Initial loss sanity:")
# with torch.no_grad():
#     pred  = model(diff_b.to(cfg.DEVICE), raw_b.to(cfg.DEVICE))
#     L, c  = total_loss(pred, sig_b.to(cfg.DEVICE), mask_b.to(cfg.DEVICE), cfg, epoch=0)
# print(f"   Total loss   : {L.item():.4f}")
# for k, v in c.items():
#     vv = v.item() if torch.is_tensor(v) else v
#     print(f"   {k:12s} : {vv:.4f}")

# print("\\n✅  Sanity checks complete. Proceed to training if all values look reasonable.")
# print("=" * 60)



print("=" * 60)
print("  PRE-TRAINING SANITY CHECKS")
print("=" * 60)


# ─────────────────────────────────────────────────────────────
# 1. Load batch
# ─────────────────────────────────────────────────────────────

diff_b, raw_b, sig_b, mask_b = next(iter(train_loader))

print("\n1. Tensor shapes from DataLoader:")

print(
    f"   diff_frames : {tuple(diff_b.shape)} "
    f"(expect: B={cfg.BATCH_SIZE}, T={cfg.SEQ_LEN}, H={cfg.IMG_SIZE}, W={cfg.IMG_SIZE}, C=3)"
)

print(f"   raw_frames  : {tuple(raw_b.shape)}")
print(f"   signal      : {tuple(sig_b.shape)}")
print(f"   mask        : {tuple(mask_b.shape)}")

print(
    f"   diff range  : [{diff_b.min().item():.3f}, {diff_b.max().item():.3f}]"
)

print(
    f"   signal std  : {sig_b.std().item():.3f} (expect ~0.9–1.1)"
)

print(
    f"   mask sum    : {mask_b.sum(dim=1).float().mean().item():.1f} valid frames avg"
)


# ─────────────────────────────────────────────────────────────
# 2. GT BPM distribution
# ─────────────────────────────────────────────────────────────

print("\n2. GT BPM distribution (first 20 windows):")

bpm_vals = []

max_samples = min(20, len(train_dataset))

for i in range(max_samples):

    try:
        _, _, sig, msk = train_dataset[i]

        valid_len = int(msk.sum().item())

        if valid_len > 32:

            sig_np = sig[:valid_len].detach().cpu().numpy()

            bpm_vals.append(compute_bpm(sig_np))

    except Exception as e:
        print(f"   ⚠️ sample {i} failed: {e}")


if len(bpm_vals) > 0:

    bpm_arr = np.asarray(bpm_vals)

    print(f"   BPM range   : [{bpm_arr.min():.1f}, {bpm_arr.max():.1f}] BPM")
    print(f"   BPM mean    : {bpm_arr.mean():.1f} BPM")
    print(f"   BPM std     : {bpm_arr.std():.1f} BPM")

    if bpm_arr.std() < 3.0:
        print("   ⚠️  LOW VARIANCE: possible GT alignment issue")
    else:
        print("   ✅  BPM distribution looks healthy")

else:
    print("   ⚠️  No valid BPM samples found")


# ─────────────────────────────────────────────────────────────
# 3. Model forward pass
# ─────────────────────────────────────────────────────────────

print("\n3. Model forward pass:")

model.eval()

with torch.no_grad():

    out = model(
        diff_b[:1].to(cfg.DEVICE),
        raw_b[:1].to(cfg.DEVICE),
    )

print(f"   Output shape : {tuple(out.shape)}")

print(f"   Output std   : {out.std().item():.4f}")

print(
    f"   Output range : "
    f"[{out.min().item():.3f}, {out.max().item():.3f}]"
)

model.train()


# ─────────────────────────────────────────────────────────────
# 4. Loss sanity
# ─────────────────────────────────────────────────────────────

print("\n4. Initial loss sanity:")

with torch.no_grad():

    pred = model(
        diff_b.to(cfg.DEVICE),
        raw_b.to(cfg.DEVICE),
    )

    L, c = total_loss(
        pred,
        sig_b.to(cfg.DEVICE),
        mask_b.to(cfg.DEVICE),
        cfg,
        epoch=0,
    )

print(f"   Total loss : {L.item():.4f}")

for k, v in c.items():

    val = v.item() if torch.is_tensor(v) else float(v)

    print(f"   {k:12s} : {val:.4f}")


print("\n✅ Sanity checks complete. Proceed if values look reasonable.")
print("=" * 60)




  PRE-TRAINING SANITY CHECKS


NameError: name 'augment_sequence' is not defined

In [ ]:
best_mae      = float("inf")
best_epoch    = 0
no_improve    = 0
train_history = []
val_history   = []

print(f"Starting training  {cfg.EPOCHS} epochs  device={cfg.DEVICE.upper()}")
print(f"  Warmup: {cfg.WARMUP_EPOCHS} epochs  |  Early stop: {cfg.EARLY_STOP} epochs")
print(f"  Curriculum: spectral+SNR losses added after epoch {cfg.WARMUP_EPOCHS}")
print("─" * 70)

for epoch in range(cfg.EPOCHS):
    t0 = time.time()

    train_m = train_one_epoch(train_loader, epoch)
    scheduler.step()
    current_lr = optimizer.param_groups[-1]['lr']
    train_history.append(train_m)

    val_m = (validate(val_loader, epoch)
             if len(val_loader) > 0
             else {'val_loss':float('nan'),'mae_bpm':float('nan'),
                   'rmse_bpm':float('nan'),'pearson_r':float('nan'),
                   'snr_db':float('nan'),'collapse':True})
    val_history.append(val_m)

    elapsed = time.time() - t0
    mae     = val_m.get('mae_bpm', float('inf'))
    col     = val_m.get('collapse', False)
    col_str = " [COLLAPSE]" if col else ""

    print(
        f"Ep {epoch+1:3d}/{cfg.EPOCHS}"
        f" | TLoss {train_m['total']:.4f}"
        f" | VLoss {val_m.get('val_loss',float('nan')):.4f}"
        f" | MAE {mae:.2f}"
        f" | RMSE {val_m.get('rmse_bpm',float('nan')):.2f}"
        f" | r={val_m.get('pearson_r',float('nan')):.3f}"
        f" | SNR {val_m.get('snr_db',float('nan')):.1f}dB"
        f" | LR {current_lr:.2e}"
        f" | {elapsed:.0f}s{col_str}"
    )

    # ── CSV ──────────────────────────────────────────────────────────────
    with open(log_path, 'a', newline='') as f:
        csv.writer(f).writerow([
            epoch+1,
            f"{train_m['total']:.6f}",
            f"{val_m.get('val_loss',''):.6f}",
            f"{val_m.get('mae_bpm',''):.4f}",
            f"{val_m.get('rmse_bpm',''):.4f}",
            f"{val_m.get('pearson_r',''):.4f}",
            f"{val_m.get('snr_db',''):.4f}",
            f"{current_lr:.8f}",
            str(val_m.get('collapse',False)),
        ])

    # ── Checkpoint ───────────────────────────────────────────────────────
    if mae < best_mae:
        best_mae   = mae
        best_epoch = epoch + 1
        no_improve = 0
        save_path  = Path(cfg.CKPT_DIR) / cfg.SAVE_BEST
        torch.save({
            'epoch': epoch+1, 'model_state': model.state_dict(),
            'ema_state': ema.shadow, 'optim_state': optimizer.state_dict(),
            'best_mae': best_mae, 'cfg': vars(cfg),
        }, save_path)
        print(f"           ✅  Best MAE {best_mae:.2f} BPM → saved {save_path.name}")
    else:
        no_improve += 1

    if no_improve >= cfg.EARLY_STOP:
        print(f"\\n⏹  Early stop ({cfg.EARLY_STOP} epochs no improvement).")
        break

# ── Save final ───────────────────────────────────────────────────────────
final_path = Path(cfg.CKPT_DIR) / cfg.SAVE_FINAL
torch.save({'epoch': epoch+1, 'model_state': model.state_dict(),
            'ema_state': ema.shadow, 'cfg': vars(cfg)}, final_path)
print(f"\\nTraining complete.")
print(f"  Best MAE   : {best_mae:.2f} BPM  (epoch {best_epoch})")
print(f"  Saved best : {Path(cfg.CKPT_DIR)/cfg.SAVE_BEST}")
print(f"  Saved final: {final_path}")


In [ ]:
def plot_training_curves(train_history, val_history):
    epochs = list(range(1, len(train_history)+1))
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle("rPPG Transformer v5 — Training Curves", fontsize=14, fontweight='bold')

    # Loss
    ax = axes[0,0]
    ax.plot(epochs, [h['total'] for h in train_history], label='Train', lw=2)
    ax.plot(epochs, [h.get('val_loss',np.nan) for h in val_history], label='Val', lw=2, ls='--')
    ax.set_title("Total Loss"); ax.legend(); ax.grid(True,alpha=0.3)
    ax.axvline(cfg.WARMUP_EPOCHS, color='gray', ls=':', lw=1, label='Warmup end')

    # Loss components
    ax = axes[0,1]
    for k, col in [('pearson','tab:blue'),('smooth_l1','tab:orange'),
                   ('spectral','tab:green'),('snr','tab:red')]:
        ax.plot(epochs, [h.get(k,np.nan) for h in train_history], label=k, lw=1.5, color=col)
    ax.set_title("Loss Components (Train)"); ax.legend(fontsize=8); ax.grid(True,alpha=0.3)
    ax.axvline(cfg.WARMUP_EPOCHS, color='gray', ls=':', lw=1)

    # MAE BPM
    ax = axes[0,2]
    mae_v = [h.get('mae_bpm',np.nan) for h in val_history]
    ax.plot(epochs, mae_v, lw=2, color='tab:red')
    valid_mae = [v for v in mae_v if not np.isnan(v)]
    if valid_mae:
        ax.axhline(min(valid_mae), color='grey', ls=':', lw=1, label=f'Best={min(valid_mae):.2f}')
    ax.set_title("Val MAE BPM"); ax.legend(); ax.grid(True,alpha=0.3)

    # RMSE
    ax = axes[1,0]
    ax.plot(epochs, [h.get('rmse_bpm',np.nan) for h in val_history], lw=2, color='tab:purple')
    ax.set_title("Val RMSE BPM"); ax.grid(True,alpha=0.3)

    # Pearson
    ax = axes[1,1]
    ax.plot(epochs, [h.get('pearson_r',np.nan) for h in val_history], lw=2, color='tab:green')
    ax.set_ylim([-0.1, 1.1])
    ax.set_title("Val Pearson r"); ax.grid(True,alpha=0.3)

    # SNR
    ax = axes[1,2]
    ax.plot(epochs, [h.get('snr_db',np.nan) for h in val_history], lw=2, color='tab:brown')
    ax.axhline(0, color='black', ls='--', lw=0.5, label='0 dB')
    ax.set_title("Val SNR (dB)"); ax.legend(fontsize=8); ax.grid(True,alpha=0.3)

    for ax in axes.flatten():
        ax.set_xlabel("Epoch")

    plt.tight_layout()
    out = Path(cfg.CKPT_DIR)/"training_curves_v5.png"
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Saved → {out}")

plot_training_curves(train_history, val_history)


In [ ]:
def bland_altman_plot(pred_bpms, gt_bpms, title="Bland-Altman"):
    means = (pred_bpms + gt_bpms) / 2
    diffs = pred_bpms - gt_bpms
    bias  = diffs.mean(); std = diffs.std(); loa = 1.96 * std

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    ax = axes[0]
    ax.scatter(gt_bpms, pred_bpms, alpha=0.7, edgecolors='k', linewidths=0.4, s=50)
    lims = [min(gt_bpms.min(), pred_bpms.min())-5, max(gt_bpms.max(), pred_bpms.max())+5]
    ax.plot(lims, lims, 'r--', lw=1.5, label='Identity')
    ax.set_xlabel("GT BPM"); ax.set_ylabel("Predicted BPM")
    ax.set_title(f"Scatter  MAE={np.abs(diffs).mean():.2f} BPM")
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.scatter(means, diffs, alpha=0.7, edgecolors='k', linewidths=0.4, s=50)
    ax.axhline(bias,       color='blue', lw=2,   ls='-',  label=f'Bias={bias:.2f}')
    ax.axhline(bias+loa,   color='red',  lw=1.5, ls='--', label=f'+1.96σ={bias+loa:.2f}')
    ax.axhline(bias-loa,   color='red',  lw=1.5, ls='--', label=f'-1.96σ={bias-loa:.2f}')
    ax.set_xlabel("Mean BPM"); ax.set_ylabel("Diff (Pred−GT)")
    ax.set_title(f"Bland-Altman  Bias={bias:.2f}  LoA=±{loa:.2f}")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    out = Path(cfg.CKPT_DIR)/"bland_altman_v5.png"
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()


def plot_signal_comparison(pred_signal, gt_signal, fs=30.0, title="Signal Comparison"):
    N   = min(len(pred_signal), len(gt_signal))
    t   = np.arange(N) / fs
    p   = bandpass_filter(pred_signal[:N], fs)
    g   = bandpass_filter(gt_signal[:N],   fs)
    f   = np.fft.rfftfreq(N, d=1.0/fs) * 60
    P_f = np.abs(np.fft.rfft(p * np.hanning(N)))
    G_f = np.abs(np.fft.rfft(g * np.hanning(N)))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(title, fontsize=12, fontweight='bold')
    axes[0].plot(t, g, lw=1.5, label='GT BVP'); axes[0].plot(t, p, lw=1.5, ls='--', label='Predicted')
    axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("Normalised Signal")
    axes[0].set_title("Time Domain"); axes[0].legend(); axes[0].grid(True,alpha=0.3)

    bm = (f >= BPM_LOW) & (f <= BPM_HIGH)
    axes[1].plot(f[bm], G_f[bm], lw=2, label='GT'); axes[1].plot(f[bm], P_f[bm], lw=2, ls='--', label='Pred')
    bg = compute_bpm(g, fs); bp = compute_bpm(p, fs)
    axes[1].axvline(bg, color='blue',   ls=':', lw=1.5, label=f'GT={bg:.1f}')
    axes[1].axvline(bp, color='orange', ls=':', lw=1.5, label=f'Pred={bp:.1f}')
    axes[1].set_xlabel("BPM"); axes[1].set_ylabel("FFT Magnitude")
    axes[1].set_title("Frequency Domain"); axes[1].legend(fontsize=8); axes[1].grid(True,alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(cfg.CKPT_DIR)/"signal_comparison_v5.png", dpi=120, bbox_inches='tight')
    plt.show()


print("Visualisation functions ready.")


In [ ]:
best_ckpt = Path(cfg.CKPT_DIR) / cfg.SAVE_BEST
if best_ckpt.exists():
    ckpt = torch.load(str(best_ckpt), map_location=cfg.DEVICE)
    if 'ema_state' in ckpt:
        model.load_state_dict({k: v.to(cfg.DEVICE) for k,v in ckpt['ema_state'].items()})
        print(f"Loaded EMA weights from {best_ckpt.name}  (epoch {ckpt.get('epoch','?')})")
    else:
        model.load_state_dict(ckpt['model_state'])
        print(f"Loaded model weights from {best_ckpt.name}")
else:
    print(f"⚠️  No checkpoint found at {best_ckpt}. Using current model weights.")

model.eval()
all_preds, all_gts = [], []

with torch.no_grad():
    for diff, raw, sig, mask in tqdm(val_loader, desc="Final eval"):
        diff = diff.to(cfg.DEVICE); raw  = raw.to(cfg.DEVICE)
        sig  = sig.to(cfg.DEVICE);  mask = mask.to(cfg.DEVICE)
        with autocast(enabled=cfg.USE_AMP):
            pred = model(diff, raw, src_key_padding_mask=(mask==0))
        for b in range(pred.shape[0]):
            vl = int(mask[b].sum().item())
            if vl > 32:
                all_preds.append(pred[b,:vl].cpu().numpy())
                all_gts.append(sig[b,:vl].cpu().numpy())

if len(all_preds) == 0:
    print("⚠️  No valid predictions. Check validation set.")
else:
    check_prediction_collapse(all_preds)  # diagnostic
    min_len   = min(len(p) for p in all_preds)
    preds_arr = np.stack([p[:min_len] for p in all_preds])
    gts_arr   = np.stack([g[:min_len] for g in all_gts])
    fm        = compute_metrics(preds_arr, gts_arr, cfg.FPS)

    print("\\n" + "="*55)
    print("  FINAL EVALUATION (Best EMA Checkpoint)")
    print("="*55)
    print(f"  MAE BPM    : {fm['mae_bpm']:.2f} BPM")
    print(f"  RMSE BPM   : {fm['rmse_bpm']:.2f} BPM")
    print(f"  Pearson r  : {fm['pearson_r']:.4f}")
    print(f"  Mean SNR   : {fm['snr_db']:.2f} dB")
    print("="*55)

    bland_altman_plot(fm['pred_bpms'], fm['gt_bpms'],
                      title="Bland-Altman — v5 Best EMA Model")
    plot_signal_comparison(all_preds[0], all_gts[0], cfg.FPS,
                           title="Sample 0 — Signal Comparison (Best EMA)")


## 🔬 Debug Cell — Inspect GT Alignment for a Single Video
Run this if metrics are still poor to verify FIX #1 is working correctly.


In [ ]:
def debug_gt_alignment(video_path: str, gt_path: str, fs: float = 30.0) -> None:
    \"\"\"
    Visualise GT alignment for one video.
    Compares old method (uniform resample) vs new method (index-based sampling).
    \"\"\"
    print(f"Debugging: {Path(video_path).parent.name}")

    diff_arr, raw_arr, frame_indices, total_frames, n_valid = process_video(
        video_path, use_cache=False
    )
    n_frames = n_valid - 1   # after temporal diff

    # Load GT raw
    raw_gt = np.loadtxt(gt_path, dtype=np.float64)
    bvp_raw = raw_gt[:,0].astype(np.float32) if raw_gt.ndim==2 else raw_gt.astype(np.float32)
    if len(bvp_raw) != total_frames:
        bvp_raw = scipy_resample(bvp_raw, total_frames).astype(np.float32)

    # OLD method: uniform resample
    gt_old = scipy_resample(bvp_raw, n_frames).astype(np.float32)
    gt_old = bandpass_filter(scipy_detrend(gt_old), fs=fs)

    # NEW method: index-based sampling
    gt_new = load_ubfc_gt_indexed(gt_path, frame_indices, total_frames, fs)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(f"GT Alignment Debug — {Path(video_path).parent.name}", fontsize=12)

    T_plot = min(300, n_frames)
    t      = np.arange(T_plot) / fs

    ax = axes[0,0]
    ax.plot(t, gt_old[:T_plot], lw=1.5, label='Old (uniform resample)', color='red', alpha=0.7)
    ax.plot(t, gt_new[:T_plot], lw=1.5, label='New (index-based)', color='blue', alpha=0.9)
    ax.set_title("GT Signal Comparison (first 10s)"); ax.legend(); ax.grid(True,alpha=0.3)
    ax.set_xlabel("Time (s)")

    ax = axes[0,1]
    # Frame drop pattern
    gaps  = np.diff(frame_indices)
    ax.plot(gaps[:T_plot], lw=1, color='purple')
    ax.axhline(1, color='green', ls='--', lw=1, label='No gap (gap=1)')
    ax.set_title(f"Frame Index Gaps (should be mostly 1)\nSkipped frames: {(gaps>1).sum()}")
    ax.legend(); ax.grid(True,alpha=0.3)

    ax = axes[1,0]
    # BPM from old vs new GT
    bpm_old = compute_bpm(gt_old, fs)
    bpm_new = compute_bpm(gt_new, fs)
    ax.text(0.2, 0.6, f"Old GT BPM: {bpm_old:.1f}", transform=ax.transAxes, fontsize=14)
    ax.text(0.2, 0.4, f"New GT BPM: {bpm_new:.1f}", transform=ax.transAxes, fontsize=14)
    ax.text(0.2, 0.2, f"HR from GT col-2 (spot check): {float(raw_gt[:,2].mean()):.1f} BPM",
            transform=ax.transAxes, fontsize=12, color='grey')
    ax.set_title("BPM Comparison"); ax.axis('off')

    ax = axes[1,1]
    N = min(len(gt_new), len(gt_old)); f_axis = np.fft.rfftfreq(N, 1/fs)*60
    mask_b = (f_axis>=BPM_LOW)&(f_axis<=BPM_HIGH)
    ax.plot(f_axis[mask_b], np.abs(np.fft.rfft(gt_old[:N]*np.hanning(N)))[mask_b], label='Old', color='red')
    ax.plot(f_axis[mask_b], np.abs(np.fft.rfft(gt_new[:N]*np.hanning(N)))[mask_b], label='New', color='blue')
    ax.set_title("FFT Spectrum (BPM band)"); ax.legend(); ax.grid(True,alpha=0.3)
    ax.set_xlabel("BPM")

    plt.tight_layout()
    plt.savefig(Path(cfg.CKPT_DIR)/"debug_gt_alignment.png", dpi=100, bbox_inches='tight')
    plt.show()

# Run on first training subject:
if train_vp:
    debug_gt_alignment(train_vp[0], train_gp[0], cfg.FPS)


In [ ]:
@torch.no_grad()
def rppg_inference(source, model=None, fs=30.0, seq_len=None, device=None, verbose=True):
    \"\"\"
    End-to-end rPPG inference on a video file or webcam.
    source : str path or int webcam index
    \"\"\"
    import cv2 as _cv2
    _model   = model  or globals()['model']
    _device  = device or cfg.DEVICE
    _seq     = seq_len or cfg.SEQ_LEN
    _model.eval().to(_device)

    face_mesh  = get_face_mesh()
    stabiliser = LandmarkStabiliser(alpha=0.65)
    cap        = _cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open: {source}")

    raw_buf, diff_buf   = [], []
    all_pred_signal     = []
    prev_frame          = None

    while True:
        ret, frame_bgr = cap.read()
        if not ret: break
        frame_rgb = _cv2.cvtColor(frame_bgr, _cv2.COLOR_BGR2RGB)
        results   = face_mesh.process(frame_rgb)
        if not results.multi_face_landmarks: continue
        lm = results.multi_face_landmarks[0]
        rois = []
        for ridx, idxs in enumerate([FOREHEAD_IDXS, LEFT_CHEEK, RIGHT_CHEEK]):
            roi, _ = extract_roi(frame_rgb, lm, idxs,
                                  stabiliser=stabiliser, roi_idx=ridx, img_size=cfg.IMG_SIZE)
            if roi is not None: rois.append(roi)
        if not rois: continue
        face = np.mean(np.stack(rois), axis=0).astype(np.float32)
        raw_buf.append(face)
        if prev_frame is not None:
            d = face - prev_frame
            diff_buf.append(((d - d.mean())/(d.std()+1e-8)).astype(np.float32))
        prev_frame = face
        if len(diff_buf) == _seq:
            dt = torch.tensor(np.stack(diff_buf)[None], dtype=torch.float32).to(_device)
            rt = torch.tensor(np.stack(raw_buf[1:_seq+1])[None], dtype=torch.float32).to(_device)
            with autocast(enabled=cfg.USE_AMP):
                pred = _model(dt, rt).squeeze(0).cpu().numpy()
            all_pred_signal.extend(pred.tolist())
            step = _seq // 2
            diff_buf = diff_buf[step:]; raw_buf = raw_buf[step:]
    cap.release()

    if len(all_pred_signal) < 16:
        return {'bpm':75.0,'signal':np.array([]),'bpm_conf':0.0,'snr_db':-99.0}
    sig = bandpass_filter(np.array(all_pred_signal,dtype=np.float32), fs)
    bpm = compute_bpm(sig, fs)
    snr = compute_snr(sig, fs)
    freqs   = np.fft.rfftfreq(len(sig), 1/fs)
    fft_mag = np.abs(np.fft.rfft(sig * np.hanning(len(sig))))
    band    = (freqs >= FREQ_LOW) & (freqs <= FREQ_HIGH)
    conf    = min(float(fft_mag[band].max() / (fft_mag[band].mean()+1e-8)) / 10.0, 1.0)
    if verbose:
        print(f"BPM: {bpm:.1f}  conf={conf:.2f}  SNR={snr:.1f}dB  frames={len(sig)}")
    return {'bpm':bpm,'signal':sig,'bpm_conf':conf,'snr_db':snr}

print("rppg_inference() ready.")
print("  result = rppg_inference('path/to/video.avi')")
print("  result = rppg_inference(0)  # webcam")


## 📋 Bug Fix Reference Card

```
╔══════════════════════════════════════════════════════════════════════════╗
║  v5 BUG FIX REFERENCE — WHY MAE WAS STUCK AT 43.74 BPM                 ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  ROOT CAUSE CHAIN:                                                       ║
║  1. GT resampled assuming uniform frames  →  timestamps misaligned       ║
║  2. Phase-shift augmentation rolled signal ±3 frames  →  100ms desync   ║
║  3. Pearson loss computed over padded zeros  →  wrong gradient direction ║
║  4. Model couldn't learn  →  outputs flat/random signal                  ║
║  5. FFT of flat signal  →  lowest BPM-band bin = 42.19 BPM always       ║
║  6. mean(|42.19 - GT_bpms|) for UBFC subjects  ≈  43.74 BPM CONSTANT   ║
║                                                                          ║
║  ADDITIONAL AMPLIFIERS:                                                  ║
║  • WARMUP_EPOCHS=5 with EPOCHS=5  →  model never reached cosine phase   ║
║  • snr_aware_loss on padded signal  →  pushed model toward flat output   ║
║  • SEQ_LEN=256 on CPU  →  too slow, too few training iterations         ║
║  • D_MODEL=128, 4 layers on CPU  →  complex model with few epochs       ║
║                                                                          ║
║  v5 FIXES APPLIED:                                                       ║
║  ✅ GT sampled at exact detected-frame indices  (FIX #1)                 ║
║  ✅ Phase-shift removed from augmentation  (FIX #2)                      ║
║  ✅ Pearson masked mean (valid frames only)  (FIX #3)                    ║
║  ✅ WARMUP_EPOCHS = max(1, EPOCHS//10)  (FIX #4)                         ║
║  ✅ Spectral/SNR losses masked before FFT  (FIX #6)                      ║
║  ✅ scipy.detrend replaces O(N³) matrix  (FIX #7)                        ║
║  ✅ Contrastive guard for B<2  (FIX #8)                                  ║
║  ✅ EMA_DECAY=0.99 on CPU  (FIX #9)                                       ║
║  ✅ CPU-adaptive model: SEQ=128,IMG=32,D=64,L=2  (FIX #10)              ║
║  ✅ SNR curriculum-gated, weight=0.05  (FIX #11)                         ║
║  ✅ EPOCHS=50 on CPU  (FIX #12)                                           ║
║                                                                          ║
║  EXPECTED BEHAVIOUR AFTER FIX:                                           ║
║  • MAE should VARY across epochs (not stuck at 43.74)                    ║
║  • Pearson r should rise above 0.01 within 5 epochs                     ║
║  • SNR should remain > -20 dB (not decrease each epoch)                 ║
║  • BPM predictions should span [50,120] range, not all = 42.19          ║
╚══════════════════════════════════════════════════════════════════════════╝
```
